# Team ReActive - MORICHIKA training pipeline

This notebook records the **actual guarded Gemma 4 E4B QLoRA development
pipeline** used in the project. It is supplied as reproducibility evidence for
the onsite submission. The final inference system uses Gemma 4 31B for
unresolved residuals; this training program developed the compact contextual
adapter and, more importantly, measured where additional fine-tuning helped or
regressed.

The verification policy was **agent-derived** from large-scale corpus exposure,
residual clustering, counterexamples, replay tests, and ablations. Its named
families are post-hoc descriptions of the regularities the agent discovered.

Training evidence spans H100 resource/quality pilots and the H200 controlled
continuation sweep under fail-closed Modal executors. The exported inference
adapter was later used with a two-T4 offline runtime. This Kaggle copy is
intentionally non-launching: credentials, private
corpora, model weights, authorization receipts, and paid execution controls are
not embedded.

## Frozen recipe and measured result

| Component | Frozen value |
|---|---|
| Base | Gemma 4 E4B QAT, pinned revision |
| Adaptation | rank-16 QLoRA; 258 target modules; 34,881,536 trainable parameters |
| Context | 4,096 tokens |
| Effective batch | 8: seven admitted gold/policy rows + one bounded synthetic row |
| Verdicts | supported / refuted / not-enough-information |
| Sweep | LR {1e-6, 2e-6} x verdict-token loss mass {0.25, 0.45} |
| Selection | frozen source/group/calibration/long-context panels; best-not-last |
| Best continuation | LR 2e-6, verdict mass 0.45, step 10 |
| Group/source mean | 0.8574 to 0.9237 |
| Long-context macro-F1 | 0.3244 to 0.5704 |
| Stop evidence | step 20 regressed; both retained arms stopped at step 40 |

The training result is a sensitivity study, not a claim that supervised
fine-tuning alone solves hallucination. Production gains came primarily from
evidence routing, retrieval, provenance, prompt structure, and selective model
allocation.

In [ ]:
from __future__ import annotations

import hashlib
import json
from pathlib import Path

SEED = 20260720
MODEL_ID = "google/gemma-4-E4B-it-qat-q4_0-unquantized"
MODEL_REVISION = "ddb04d7360c7d7c353532e0e797038439dd1738a"
MAX_SEQUENCE_LENGTH = 4096
LORA_RANK = 16
EXPECTED_TARGET_MODULES = 258
EXPECTED_TRAINABLE_PARAMETERS = 34_881_536
EFFECTIVE_BATCH = 8
GOLD_PER_STEP = 7
SYNTHETIC_PER_STEP = 1
VERDICTS = ("supported", "refuted", "not_enough_information")
ARMS = {
    "lr1e6_verdict25": {"learning_rate": 1e-6, "verdict_loss_mass": 0.25},
    "lr1e6_verdict45": {"learning_rate": 1e-6, "verdict_loss_mass": 0.45},
    "lr2e6_verdict25": {"learning_rate": 2e-6, "verdict_loss_mass": 0.25},
    "lr2e6_verdict45": {"learning_rate": 2e-6, "verdict_loss_mass": 0.45},
}

def sha256(path: str | Path) -> str:
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(8 * 1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

print({
    "model": MODEL_ID,
    "revision": MODEL_REVISION,
    "arms": ARMS,
    "selection": "best-not-last",
    "automatic_paid_launch": False,
})

Path("training_code").mkdir(exist_ok=True)
print("Training sources will be exported to ./training_code by the next cells.")

## Sealed curriculum builder

The following code cell is the exact source used to construct the fair 2x2
continuation pack. Each arm receives the same immutable 800-record order and
100 optimizer-step schedule. Corpus rows are admitted by rights, overlap,
lineage, tokenization, source, and verdict checks before this stage. Executing
the cell exports the source file; it does not launch training.

In [ ]:
%%writefile training_code/build_gemma4_e4b_step50_continuation_sweep_v1.py
"""Seal the fair 2x2 Gemma step-50 continuation sweep.

Every effective batch is exactly seven canonical-gold parents plus one
rights/overlap/token-audited owned indirect-context parent.  One immutable
schedule is shared by all arms.  This local builder cannot launch Modal.
"""

from __future__ import annotations

from collections import Counter, defaultdict
from collections.abc import Mapping, Sequence
import argparse
import hashlib
import json
import math
from pathlib import Path
import random
import shutil
from typing import Any
import uuid

from pipeline import build_gemma4_e4b_contextual_repair_factorial_canary_v1 as repair


ROOT = Path(__file__).resolve().parents[1]
GOLD = ROOT / "artifacts/training/gemma4_e4b_step50_guarded_continuation_v1_20260720"
SYNTH = ROOT / "artifacts/training/gemma4_e4b_owned_indirect_context_candidate_v1_20260720"
SYNTH_AUDIT = ROOT / "artifacts/status/OWNED_INDIRECT_CONTEXT_CANDIDATE_INDEPENDENT_AUDIT_20260720.md"
OUTPUT = ROOT / "artifacts/training/gemma4_e4b_step50_continuation_sweep_v1_20260720"

VERSION = "bichar-gemma4-e4b-step50-continuation-sweep-pack-v1"
SEED = "20260720:gemma-e4b:step50:canonical7-synthetic1:identical-arms:v1"
VERDICTS = ("supported", "refuted", "not_enough_information")
ARMS = {
    "lr1e6_verdict25": {"base_learning_rate": 1e-6, "verdict_loss_mass": 0.25},
    "lr2e6_verdict25": {"base_learning_rate": 2e-6, "verdict_loss_mass": 0.25},
    "lr1e6_verdict45": {"base_learning_rate": 1e-6, "verdict_loss_mass": 0.45},
    "lr2e6_verdict45": {"base_learning_rate": 2e-6, "verdict_loss_mass": 0.45},
}
TOTAL_STEPS = 100
EFFECTIVE_BATCH = 8
GOLD_PER_STEP = 7
SYNTH_PER_STEP = 1

GOLD_MANIFEST_SHA256 = "ce0f8eed7141b7fa7041ca6bd53d453920cac721394dbd570dae6172aa347866"
GOLD_MANIFEST_ID = "b3a54d021b0c8d3e4da111aef94ada24e3325d1225b7b18c1ff8fc04a24ccd8b"
SYNTH_MANIFEST_SHA256 = "f6467297c198c8b8c6371a4697120901df3ba235ee8e0985ef3836d891c45d8e"
SYNTH_MANIFEST_ID = "7ef63b3df138fb0e11a574ee7df02cddf6afa994012152dcb6633455e5b63fad"
SYNTH_AUDIT_SHA256 = "8fe6290dc8e35686766d5d70f59002bf58c5a31d19e8c72d9dcda3210f61792e"


class BuildError(ValueError):
    pass


def canonical(value: Any) -> str:
    return json.dumps(value, ensure_ascii=False, sort_keys=True, separators=(",", ":"))


def sha_bytes(value: bytes) -> str:
    return hashlib.sha256(value).hexdigest()


def sha_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(8 * 1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def contract(value: Mapping[str, Any], field: str) -> dict[str, Any]:
    body = dict(value)
    body.pop(field, None)
    body[field] = sha_bytes(canonical(body).encode("utf-8"))
    return body


def valid_contract(value: Mapping[str, Any], field: str) -> bool:
    return isinstance(value.get(field), str) and contract(value, field)[field] == value[field]


def read_json(path: Path, expected: str | None = None) -> dict[str, Any]:
    if expected is not None and sha_file(path) != expected:
        raise BuildError(f"hash drifted: {path}")
    value = json.loads(path.read_text(encoding="utf-8"))
    if not isinstance(value, dict):
        raise BuildError(f"object expected: {path}")
    return value


def read_jsonl(path: Path) -> list[dict[str, Any]]:
    return [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]


def display(path: Path) -> str:
    try:
        return path.resolve().relative_to(ROOT.resolve()).as_posix()
    except ValueError:
        return path.resolve().as_posix()


def write_json(path: Path, value: Mapping[str, Any]) -> dict[str, Any]:
    path.write_text(json.dumps(value, ensure_ascii=False, sort_keys=True, indent=2) + "\n", encoding="utf-8")
    return {"path": display(path), "bytes": path.stat().st_size, "sha256": sha_file(path)}


def write_jsonl(path: Path, rows: Sequence[Mapping[str, Any]]) -> dict[str, Any]:
    with path.open("w", encoding="utf-8", newline="\n") as stream:
        for row in rows:
            stream.write(canonical(row) + "\n")
    return {"path": display(path), "rows": len(rows), "bytes": path.stat().st_size, "sha256": sha_file(path)}


def learning_rate_at(step: int, base: float) -> float:
    """Warm up 1..4, hold through 60, cosine-decay 61..100 to 10%."""
    if not 1 <= step <= TOTAL_STEPS or base <= 0:
        raise BuildError("invalid scheduler input")
    if step <= 4:
        return base * step / 4.0
    if step <= 60:
        return base
    floor = 0.10 * base
    phase = (step - 60) / 40.0
    return floor + 0.5 * (base - floor) * (1.0 + math.cos(math.pi * phase))


def _verify_source_artifacts(manifest: Mapping[str, Any], root: Path, names: Sequence[str]) -> None:
    for name in names:
        spec = manifest["artifacts"][name]
        path = root / name
        if path.is_symlink() or sha_file(path) != spec["sha256"]:
            raise BuildError(f"source artifact drifted: {name}")
        if "rows" in spec and len(read_jsonl(path)) != int(spec["rows"]):
            raise BuildError(f"source row count drifted: {name}")


def _synthetic_admission(manifest: Mapping[str, Any]) -> dict[str, Any]:
    audit = read_json(SYNTH / "audit_receipt.json", manifest["artifacts"]["audit_receipt.json"]["sha256"])
    rights = read_json(SYNTH / "rights_receipt.json", manifest["artifacts"]["rights_receipt.json"]["sha256"])
    overlap = read_json(SYNTH / "overlap_audit.json", manifest["artifacts"]["overlap_audit.json"]["sha256"])
    token = read_json(SYNTH / "exact_token_audit.json", manifest["artifacts"]["exact_token_audit.json"]["sha256"])
    unicode = read_json(SYNTH / "unicode_audit.json", manifest["artifacts"]["unicode_audit.json"]["sha256"])
    if not (
        audit.get("status") == "PASS_TRAIN_ONLY_CANDIDATE_NOT_SCHEDULED"
        and audit.get("rights_passed") is True
        and audit.get("overlap_passed") is True
        and audit.get("exact_token_passed") is True
        and audit.get("unicode_passed") is True
        and audit.get("all_triads_group_closed") is True
        and rights.get("private_training_allowed_by_owner") is True
        and rights.get("competition_text_copied") is False
        and rights.get("third_party_text_copied") is False
        and overlap.get("failed_rows") == overlap.get("failed_parents") == 0
        and overlap.get("labels_read") is False
        and token.get("all_rows_at_or_below_4096") is True
        and token.get("rows_checked") == 1800
        and unicode.get("failures", unicode.get("failure_count", 0)) in (0, [])
        and sha_file(SYNTH_AUDIT) == SYNTH_AUDIT_SHA256
    ):
        raise BuildError("owned synthetic candidate is not admissible for the bounded sweep")
    return contract(
        {
            "version": VERSION,
            "status": "admitted_owned_indirect_context_for_bounded_12_5_percent_sweep_only",
            "candidate_manifest_id": SYNTH_MANIFEST_ID,
            "candidate_manifest_sha256": SYNTH_MANIFEST_SHA256,
            "independent_audit_sha256": SYNTH_AUDIT_SHA256,
            "rights_private_training_passed": True,
            "competition_text_absent": True,
            "protected_overlap_zero": True,
            "exact_gemma_tokenization_passed": True,
            "unicode_passed": True,
            "semantic_status": "conditional_pass_bounded_ablation_only",
            "maximum_mixture_fraction": 0.125,
            "maximum_selected_rows": 100,
            "maximum_selected_parents": 100,
            "heavy_backbone_admission": False,
            "promotion_authority": False,
        },
        "admission_id",
    )


def _synthetic_parent_order(rows: Sequence[Mapping[str, Any]]) -> tuple[list[str], dict[str, dict[str, dict[str, Any]]]]:
    triads: dict[str, dict[str, dict[str, Any]]] = defaultdict(dict)
    family_of: dict[str, str] = {}
    for row in rows:
        parent, verdict = str(row["parent_id"]), str(row["verdict"])
        triads[parent][verdict] = dict(row)
        family_of[parent] = str(row["family"])
    if len(triads) != 600 or any(set(values) != set(VERDICTS) for values in triads.values()):
        raise BuildError("synthetic triad closure drifted")
    by_family: dict[str, list[str]] = defaultdict(list)
    for parent in triads:
        by_family[family_of[parent]].append(parent)
    for family in by_family:
        by_family[family].sort(key=lambda parent: sha_bytes(f"{SEED}:synthetic-parent:{parent}".encode()))
    families = sorted(by_family)
    order: list[str] = []
    depth = 0
    while len(order) < 100:
        for family in families:
            if depth < len(by_family[family]):
                order.append(by_family[family][depth])
                if len(order) == 100:
                    break
        depth += 1
    if len(order) != 100 or len(set(order)) != 100:
        raise BuildError("synthetic parent order repeats")
    return order, triads


def _target_counts(step: int) -> dict[str, int]:
    rotations = (
        {"supported": 3, "refuted": 3, "not_enough_information": 2},
        {"supported": 3, "refuted": 2, "not_enough_information": 3},
        {"supported": 2, "refuted": 3, "not_enough_information": 3},
    )
    return rotations[(step - 1) % 3]


def build(output: Path = OUTPUT) -> dict[str, Any]:
    gold_manifest = read_json(GOLD / "manifest.json", GOLD_MANIFEST_SHA256)
    synth_manifest = read_json(SYNTH / "manifest.json", SYNTH_MANIFEST_SHA256)
    if not (valid_contract(gold_manifest, "manifest_id") and gold_manifest["manifest_id"] == GOLD_MANIFEST_ID):
        raise BuildError("gold manifest identity drifted")
    if not (valid_contract(synth_manifest, "manifest_id") and synth_manifest["manifest_id"] == SYNTH_MANIFEST_ID):
        raise BuildError("synthetic manifest identity drifted")
    _verify_source_artifacts(gold_manifest, GOLD, ("train_examples.jsonl", "train_encoded.jsonl", "loss_weight_spans.jsonl"))
    _verify_source_artifacts(synth_manifest, SYNTH, ("train_examples.jsonl", "train_encoded.jsonl"))
    admission = _synthetic_admission(synth_manifest)

    gold_rows = read_jsonl(GOLD / "train_examples.jsonl")
    gold_encoded = {str(row["example_id"]): row for row in read_jsonl(GOLD / "train_encoded.jsonl")}
    gold_spans = {str(row["example_id"]): row for row in read_jsonl(GOLD / "loss_weight_spans.jsonl")}
    synth_rows = read_jsonl(SYNTH / "train_examples.jsonl")
    synth_encoded = {str(row["example_id"]): row for row in read_jsonl(SYNTH / "train_encoded.jsonl")}
    parent_order, triads = _synthetic_parent_order(synth_rows)

    gold_queues: dict[str, list[dict[str, Any]]] = defaultdict(list)
    for row in gold_rows:
        gold_queues[str(row["verdict"])].append(row)
    cursors = Counter()
    selected_raw: list[dict[str, Any]] = []
    selected_encoded: list[dict[str, Any]] = []
    selected_spans: list[dict[str, Any]] = []
    steps: list[dict[str, Any]] = []
    synthetic_selection: list[dict[str, Any]] = []
    cumulative_tokens = 0

    for step in range(1, TOTAL_STEPS + 1):
        synth_verdict = VERDICTS[(step - 1) % 3]
        synth_row = triads[parent_order[step - 1]][synth_verdict]
        total_counts = _target_counts(step)
        gold_counts = dict(total_counts)
        gold_counts[synth_verdict] -= 1
        batch: list[tuple[str, dict[str, Any]]] = [("synthetic", synth_row)]
        for verdict in VERDICTS:
            start, end = cursors[verdict], cursors[verdict] + gold_counts[verdict]
            chosen = gold_queues[verdict][start:end]
            if len(chosen) != gold_counts[verdict]:
                raise BuildError("canonical queue exhausted")
            batch.extend(("canonical_gold", row) for row in chosen)
            cursors[verdict] = end
        random.Random(f"{SEED}:step:{step}").shuffle(batch)
        if not (
            len(batch) == 8
            and Counter(origin for origin, _ in batch) == Counter({"canonical_gold": 7, "synthetic": 1})
            and len({str(row["parent_id"]) for _, row in batch}) == 8
            and Counter(str(row["verdict"]) for _, row in batch) == Counter(total_counts)
        ):
            raise BuildError("7+1 batch contract drifted")
        step_ids: list[str] = []
        for slot, (origin, source_row) in enumerate(batch, 1):
            row = dict(source_row)
            row["sweep_origin"] = origin
            row["sweep_optimizer_step"] = step
            row["sweep_step_slot"] = slot
            example_id = str(row["example_id"])
            encoded = gold_encoded[example_id] if origin == "canonical_gold" else synth_encoded[example_id]
            span = gold_spans[example_id] if origin == "canonical_gold" else repair._audit_encoded_row(source_row, encoded)
            selected_raw.append(row)
            selected_encoded.append(encoded)
            selected_spans.append(span)
            step_ids.append(example_id)
        native_tokens = sum(int(encoded["native_token_count"]) for _, encoded in zip(batch, selected_encoded[-8:], strict=True))
        cumulative_tokens += native_tokens
        synthetic_selection.append({"optimizer_step": step, "parent_id": synth_row["parent_id"], "example_id": synth_row["example_id"], "family": synth_row["family"], "verdict": synth_verdict})
        steps.append(
            {
                "optimizer_step": step,
                "block_index": (step - 1) // 3 + 1,
                "example_ids": step_ids,
                "parent_ids": [str(row["parent_id"]) for _, row in batch],
                "origins": [origin for origin, _ in batch],
                "verdicts": [str(row["verdict"]) for _, row in batch],
                "synthetic_example_id": synth_row["example_id"],
                "synthetic_parent_id": synth_row["parent_id"],
                "synthetic_verdict": synth_verdict,
                "canonical_gold_rows": 7,
                "synthetic_rows": 1,
                "native_tokens": native_tokens,
                "cumulative_native_tokens": cumulative_tokens,
            }
        )

    if len({row["example_id"] for row in selected_raw}) != 800 or len({row["parent_id"] for row in selected_raw}) != 800:
        raise BuildError("sweep schedule is not unique-example and unique-parent")
    if len({row["parent_id"] for row in selected_raw if row["sweep_origin"] == "synthetic"}) != 100:
        raise BuildError("synthetic parent repeated before step 100")
    for offset in range(0, 99, 3):
        observed = Counter(v for step in steps[offset:offset + 3] for v in step["verdicts"])
        if observed != Counter({verdict: 8 for verdict in VERDICTS}):
            raise BuildError("three-step block is not exactly 8/8/8")

    scheduler_tables = {
        arm: [{"optimizer_step": step, "learning_rate": learning_rate_at(step, spec["base_learning_rate"])} for step in range(1, 101)]
        for arm, spec in ARMS.items()
    }
    spec = contract(
        {
            "version": VERSION,
            "status": "sealed_2x2_successive_halving_sweep_no_launch_authority",
            "initialization": {
                "source": "immutable_lr1e5_verdict25_step50_adapter",
                "checkpoint_marker_id": "028de29c41ec7036bf1b42ed298b9fc64584d6cecbdc02c1d8bf84bb0314e095",
                "checkpoint_marker_sha256": "7ca2cfde9ad820829ec64a1453a5b268fa0f87bc5f792517affb0eea5b9a8e36",
                "checkpoint_tree_sha256": "06057c936b3866e0fb6a18eaf455ae69945a61a1419697c7620a7218b72fb0e9",
                "adapter_tree_sha256": "9bb2d6bbc23cd90f59db501ba0b5cf84072ad87a750b3da589896ef43fb4dbf4",
                "fresh_optimizer_per_arm": True,
            },
            "arms": ARMS,
            "scheduler": {
                "formula": "step<=4:base*step/4; 5<=step<=60:base; 61<=step<=100:0.1*base+0.45*base*(1+cos(pi*(step-60)/40))",
                "warmup_steps": [1, 2, 3, 4],
                "constant_steps": [5, 60],
                "cosine_decay_steps": [61, 100],
                "terminal_fraction": 0.10,
                "tables": scheduler_tables,
            },
            "curriculum": {
                "optimizer_steps": 100,
                "effective_batch_size": 8,
                "canonical_gold_per_batch": 7,
                "owned_synthetic_per_batch": 1,
                "canonical_gold_fraction": 0.875,
                "owned_synthetic_fraction": 0.125,
                "unique_examples": 800,
                "unique_parents": 800,
                "unique_synthetic_parents_before_repeat": 100,
                "synthetic_verdict_rotation": list(VERDICTS),
                "verdict_counts": dict(Counter(row["verdict"] for row in selected_raw)),
                "each_complete_three_step_block_verdict_counts": {verdict: 8 for verdict in VERDICTS},
                "schedule_shared_identically_by_all_arms": True,
                "seed": SEED,
            },
            "successive_halving": {
                "screen": {"arms": 4, "start_step": 0, "end_step": 20, "evaluation_steps": [0, 10, 20]},
                "trend": {"arms": 2, "start_step": 20, "end_step": 60, "evaluation_steps": [30, 40, 50, 60]},
                "tail": {"arms": 1, "start_step": 60, "end_step": 100, "evaluation_steps": [70, 80, 90, 100]},
                "maximum_total_optimizer_updates": 200,
                "catastrophic_stops_immediate": True,
                "aggregate_soft_patience_starts_after_step": 20,
                "aggregate_soft_patience": 2,
                "material_group_source_mean_drop": 0.03,
                "best_not_last": True,
            },
            "execution": {
                "mode": "fresh_hash_bound_base_reload_for_each_arm_and_resume_phase_inside_same_single_use_container",
                "reason": "contamination-proof fallback because exact in-memory base invariance reuse was not proven",
                "base_reuse_enabled": False,
                "exact_offline_model_tree_verification_required_before_each_base_load": True,
                "delete_model_optimizer_trainables_and_empty_cuda_cache_after_each_phase": True,
                "maximum_base_loads": 7,
                "one_detached_campaign": True,
                "network_blocked": True,
                "max_containers": 1,
                "retries": 0,
                "hard_aggregate_compute_cap_usd": 2.75,
                "minimum_outside_credit_reserve_usd": 10.0,
                "launch_authorized": False,
                "promotion_allowed": False,
            },
        },
        "spec_id",
    )

    staging = output.parent / f".{output.name}.staging-{uuid.uuid4().hex}"
    staging.mkdir(parents=True, exist_ok=False)
    try:
        artifacts = {
            "train_examples.jsonl": write_jsonl(staging / "train_examples.jsonl", selected_raw),
            "train_encoded.jsonl": write_jsonl(staging / "train_encoded.jsonl", selected_encoded),
            "loss_weight_spans.jsonl": write_jsonl(staging / "loss_weight_spans.jsonl", selected_spans),
            "optimizer_steps.jsonl": write_jsonl(staging / "optimizer_steps.jsonl", steps),
            "synthetic_selection.jsonl": write_jsonl(staging / "synthetic_selection.jsonl", synthetic_selection),
            "synthetic_admission.json": write_json(staging / "synthetic_admission.json", admission),
            "sweep_spec.json": write_json(staging / "sweep_spec.json", spec),
        }
        # Bind paths to the final location, not the temporary staging name.
        for value in artifacts.values():
            value["path"] = f"artifacts/training/{output.name}/{Path(value['path']).name}"
        manifest = contract(
            {
                "version": VERSION,
                "status": "SEALED_LOCAL_2X2_SWEEP_PACK_NO_LAUNCH_AUTHORITY",
                "builder": {"path": "pipeline/build_gemma4_e4b_step50_continuation_sweep_v1.py", "sha256": sha_file(Path(__file__))},
                "gold_parent": {"manifest_id": GOLD_MANIFEST_ID, "manifest_sha256": GOLD_MANIFEST_SHA256},
                "synthetic_parent": {"manifest_id": SYNTH_MANIFEST_ID, "manifest_sha256": SYNTH_MANIFEST_SHA256, "admission_id": admission["admission_id"]},
                "artifacts": artifacts,
                "panel_bindings": gold_manifest["panel_bindings"],
                "private_competition_labels_included": False,
                "launch_authorized": False,
                "paid_action_performed": False,
                "promotion_allowed": False,
            },
            "manifest_id",
        )
        write_json(staging / "manifest.json", manifest)
        if output.exists():
            if output.resolve() != OUTPUT.resolve():
                raise BuildError("refusing to replace a noncanonical existing output directory")
            shutil.rmtree(output)
        staging.replace(output)
        return manifest
    except BaseException:
        shutil.rmtree(staging, ignore_errors=True)
        raise


def verify(output: Path = OUTPUT) -> dict[str, Any]:
    manifest = read_json(output / "manifest.json")
    if not (
        valid_contract(manifest, "manifest_id")
        and manifest.get("builder", {}).get("path") == "pipeline/build_gemma4_e4b_step50_continuation_sweep_v1.py"
        and manifest.get("builder", {}).get("sha256") == sha_file(Path(__file__))
    ):
        raise BuildError("manifest contract invalid")
    for name, artifact in manifest["artifacts"].items():
        path = output / name
        if sha_file(path) != artifact["sha256"]:
            raise BuildError(f"output drifted: {name}")
        if "rows" in artifact and len(read_jsonl(path)) != artifact["rows"]:
            raise BuildError(f"output row count drifted: {name}")
    steps = read_jsonl(output / "optimizer_steps.jsonl")
    for step in steps:
        if Counter(step["origins"]) != Counter({"canonical_gold": 7, "synthetic": 1}):
            raise BuildError("verified schedule lost 7+1")
    return {"status": "verified", "manifest_id": manifest["manifest_id"], "manifest_sha256": sha_file(output / "manifest.json"), "optimizer_steps_sha256": manifest["artifacts"]["optimizer_steps.jsonl"]["sha256"]}


def main() -> int:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--verify-only", action="store_true")
    args = parser.parse_args()
    value = verify() if args.verify_only else build()
    print(json.dumps(value, ensure_ascii=False, sort_keys=True, indent=2))
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


## Guarded QLoRA training loop

This is the actual guarded training implementation: NF4 loading, PEFT adapter
binding, fresh AdamW state, weighted causal-verdict loss, gradient
accumulation, clipping, VRAM/OOM gates, atomic checkpoints, frozen evaluation
panels, regression stops, and best-not-last retention. The code is visible as
code and is exported as a `.py` file; the paid authorization and private
bindings needed to launch it are intentionally absent.

In [ ]:
%%writefile training_code/modal_train_gemma4_e4b_step50_guarded_continuation_v1.py
"""Guarded, launch-disabled-by-default continuation of immutable Gemma step 50.

The only preregistered recipe starts from the exact lr1e5/verdict25 step-50
adapter with a fresh AdamW optimizer, 2e-6 LR, 45% verdict-span loss mass,
and two separately authorized stages (60 required + 40 optional optimizer
steps).  It evaluates every 10 steps, stops catastrophic regression at once,
uses patience two for modest regression, and records an authenticated best
checkpoint independently of the last checkpoint.

Static inspection, acceptance, planning, and authorization construction are
inert.  Only the local Modal entrypoint can spawn one paid call, and it
requires a fresh hash-bound authorization plus an explicit detach attestation.
"""

from __future__ import annotations

import argparse
import ast
from collections import Counter
from collections.abc import Mapping, Sequence
import hashlib
import json
import math
import os
from pathlib import Path
import random
import re
import shutil
import statistics
import sys
import time
import traceback
from typing import Any
import uuid


os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

import modal


ROOT = Path(__file__).resolve().parents[1]
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from pipeline import modal_train_gemma4_e4b_contextual_heavy14m_v1 as proven
from pipeline import modal_train_gemma4_e4b_contextual_repair_factorial_canary_v1 as factorial


VERSION = "bichar-gemma4-e4b-step50-guarded-continuation-executor-v1"
AUTH_VERSION = "bichar-gemma4-e4b-step50-guarded-continuation-authorization-v1"
ACCEPTANCE_VERSION = "bichar-gemma4-e4b-step50-guarded-continuation-static-acceptance-v1"
PACK = ROOT / "artifacts/training/gemma4_e4b_step50_guarded_continuation_v1_20260720"
MANIFEST = PACK / "manifest.json"
SPEC = PACK / "continuation_spec.json"
STEP50_SUMMARY = ROOT / "artifacts/status/GEMMA4_E4B_REPAIR_LR1E5_VERDICT25_R1_STEP50_SUMMARY_20260720.json"
CONFIRMATORY_DECISION = ROOT / "artifacts/status/GEMMA4_E4B_VERDICT25_CONFIRMATORY600_R1_DECISION_20260720.json"
DEFAULT_ACCEPTANCE = ROOT / "artifacts/status/GEMMA4_E4B_STEP50_GUARDED_CONTINUATION_ACCEPTANCE_20260720.json"
DEFAULT_COMMANDS = ROOT / "artifacts/status/GEMMA4_E4B_STEP50_GUARDED_CONTINUATION_SAFE_COMMANDS_20260720.json"

MANIFEST_SHA256 = "ce0f8eed7141b7fa7041ca6bd53d453920cac721394dbd570dae6172aa347866"
MANIFEST_ID = "b3a54d021b0c8d3e4da111aef94ada24e3325d1225b7b18c1ff8fc04a24ccd8b"
SPEC_SHA256 = "579945ff8a61536eabd3dbc80b9831a5bb34675506d8ec79e2fb7040f38656b5"
SPEC_ID = "6c9e4a70596e97430db8cd0583ace785d4f82214f20ccc047c880ad616552353"
STEP50_SUMMARY_SHA256 = "4b193314840bf763105590fa23688da8f180db20189aee8bc9262c15d243d5f8"
CONFIRMATORY_DECISION_SHA256 = "8eb5a8fee22483d3e2809dcbd2c519a744acb5f6969692da561b2b324ea62c6e"

SOURCE_CHECKPOINT_VOLUME = "bichar-gemma4-e4b-contextual-repair-factorial-v1"
OUTPUT_CHECKPOINT_VOLUME = "bichar-gemma4-e4b-step50-guarded-continuation-v1"
MODEL_VOLUME = factorial.MODEL_VOLUME_NAME
SOURCE_STEP50_RELATIVE = "runs/lr1e5_verdict25/gemma4_e4b_repair_lr1e5_verdict25_r1_20260720/checkpoints/step-0000050"
SOURCE_RUN_ID = "gemma4_e4b_repair_lr1e5_verdict25_r1_20260720"
SOURCE_MARKER_ID = "028de29c41ec7036bf1b42ed298b9fc64584d6cecbdc02c1d8bf84bb0314e095"
SOURCE_MARKER_SHA256 = "7ca2cfde9ad820829ec64a1453a5b268fa0f87bc5f792517affb0eea5b9a8e36"
SOURCE_CHECKPOINT_TREE_SHA256 = "06057c936b3866e0fb6a18eaf455ae69945a61a1419697c7620a7218b72fb0e9"
SOURCE_ADAPTER_TREE_SHA256 = "9bb2d6bbc23cd90f59db501ba0b5cf84072ad87a750b3da589896ef43fb4dbf4"
VERDICTS = ("supported", "refuted", "not_enough_information")
PANEL_ROLES = factorial.PANEL_ROLES
PACK_FILES = factorial.PACK_FILES
TOTAL_STEPS = 100
STAGE_A_END = 60
STAGE_B_END = 100
STAGE_IDS = ("stage_a", "stage_b")
EVALUATION_STEPS = tuple(range(0, TOTAL_STEPS + 1, 10))
TRAINING_GATE_STEPS = tuple(range(10, TOTAL_STEPS + 1, 10))
SEED = 20260720
LEARNING_RATE = 2e-6
WARMUP_STEPS = 4
VERDICT_LOSS_MASS = 0.45
MICRO_BATCH_SIZE = 4
GRADIENT_ACCUMULATION = 2
EFFECTIVE_BATCH_SIZE = 8
MAX_SEQUENCE_TOKENS = 4096
MAXIMUM_PEAK_VRAM_BYTES = factorial.MAXIMUM_PEAK_VRAM_BYTES
HARD_WALL_SECONDS = 900
MAXIMUM_TOTAL_COMPUTE_CHARGE_USD = 1.25
MAXIMUM_GPU_CHARGE_USD = 1.05
PERSISTENCE_RESERVE_USD = 0.12
MINIMUM_OUTSIDE_RESERVE_USD = 10.0
GPU_RATE_USD_PER_SECOND = factorial.GPU_RATE_USD_PER_SECOND
CPU_RATE_USD_PER_CORE_SECOND = factorial.CPU_RATE_USD_PER_CORE_SECOND
MEMORY_RATE_USD_PER_GIB_SECOND = factorial.MEMORY_RATE_USD_PER_GIB_SECOND
CPU_CORES = 8.0
MEMORY_GIB = 64.0
TOTAL_RATE_USD_PER_SECOND = GPU_RATE_USD_PER_SECOND + CPU_CORES * CPU_RATE_USD_PER_CORE_SECOND + MEMORY_GIB * MEMORY_RATE_USD_PER_GIB_SECOND
EVALUATION_RESERVE_SECONDS = 70.0
REGRESSION_TOLERANCE = 1e-12
SOFT_REGRESSION_PATIENCE = 2
SUPPORTED_RECALL_CATASTROPHIC_DROP = 0.20
TARGET_TERMINATOR_ID = factorial.TARGET_TERMINATOR_ID
TARGET_TERMINATOR_TOKEN = factorial.TARGET_TERMINATOR_TOKEN
RUN_ID_RE = re.compile(r"(?=.{1,96}$)[A-Za-z0-9](?:[A-Za-z0-9_.-]*[A-Za-z0-9])?")


class ContinuationError(ValueError):
    pass


class QualityRegressionStop(RuntimeError):
    def __init__(self, step: int, gate: Mapping[str, Any]):
        super().__init__(f"strict quality regression at optimizer step {step}")
        self.step = int(step)
        self.gate = dict(gate)


class CleanBudgetStop(RuntimeError):
    pass


canonical = factorial.canonical
sha_bytes = factorial.sha_bytes
sha_file = factorial.sha_file
content_contract = factorial.content_contract
valid_contract = factorial.valid_contract
read_json = factorial.read_json
read_jsonl = factorial.read_jsonl
atomic_json = factorial.atomic_json
write_jsonl = factorial.write_jsonl
tree_binding = factorial.tree_binding


def _local_artifact_path(spec: Mapping[str, Any]) -> Path:
    candidate = (ROOT / str(spec["path"])).resolve()
    if ROOT.resolve() not in candidate.parents:
        raise ContinuationError("artifact escapes project root")
    return candidate


def _valid_run_id(value: Any) -> bool:
    return isinstance(value, str) and RUN_ID_RE.fullmatch(value) is not None and value not in {".", ".."}


def _load_pack(*, verify_artifacts: bool = True) -> dict[str, Any]:
    manifest = read_json(MANIFEST, MANIFEST_SHA256)
    spec = read_json(SPEC, SPEC_SHA256)
    step50 = read_json(STEP50_SUMMARY, STEP50_SUMMARY_SHA256)
    decision = read_json(CONFIRMATORY_DECISION, CONFIRMATORY_DECISION_SHA256)
    if not (
        valid_contract(manifest, "manifest_id")
        and manifest["manifest_id"] == MANIFEST_ID
        and valid_contract(spec, "spec_id")
        and spec["spec_id"] == SPEC_ID
        and manifest["artifacts"]["continuation_spec.json"]["sha256"] == SPEC_SHA256
        and manifest.get("private_confirmatory600_labels_included") is False
    ):
        raise ContinuationError("sealed manifest/spec identity drifted")
    init = spec["initialization"]
    recipe = spec["single_preregistered_recipe"]
    execution = spec["execution"]
    if not (
        init["checkpoint_marker_id"] == SOURCE_MARKER_ID
        and init["checkpoint_marker_sha256"] == SOURCE_MARKER_SHA256
        and init["checkpoint_tree_sha256"] == SOURCE_CHECKPOINT_TREE_SHA256
        and init["adapter_tree_sha256"] == SOURCE_ADAPTER_TREE_SHA256
        and init["fresh_optimizer_required"] is True
        and init["old_optimizer_scheduler_rng_forbidden"] is True
        and recipe["learning_rate"] == LEARNING_RATE
        and recipe["verdict_value_loss_mass"] == VERDICT_LOSS_MASS
        and recipe["all_other_supervised_target_loss_mass"] == 1.0 - VERDICT_LOSS_MASS
        and recipe["auxiliary_objective"] is None
        and recipe["effective_batch_size"] == EFFECTIVE_BATCH_SIZE
        and spec["evaluation"]["optimizer_steps"] == list(EVALUATION_STEPS)
        and spec["evaluation"]["soft_regression_patience_consecutive_gates"] == SOFT_REGRESSION_PATIENCE
        and execution["maximum_steps"] == TOTAL_STEPS
        and execution["stages"]["stage_a"] == {"start_optimizer_step": 0, "end_optimizer_step": STAGE_A_END, "required": True}
        and execution["stages"]["stage_b"] == {"start_optimizer_step": STAGE_A_END, "end_optimizer_step": STAGE_B_END, "required": False, "requires_stage_a_boundary_stable_or_improving": True}
        and execution["fresh_one_use_cost_authorization_required_per_stage"] is True
        and execution["hard_wall_seconds"] == HARD_WALL_SECONDS
        and execution["maximum_total_compute_charge_usd"] == MAXIMUM_TOTAL_COMPUTE_CHARGE_USD
        and execution["maximum_gpu_charge_usd"] == MAXIMUM_GPU_CHARGE_USD
        and execution["persistence_reserve_usd"] == PERSISTENCE_RESERVE_USD
        and execution["minimum_outside_credit_reserve_usd"] == MINIMUM_OUTSIDE_RESERVE_USD
        and execution["launch_authorized"] is False
        and spec["optional_indirect_context_extension"]["runtime_unsealed_extension_allowed"] is False
    ):
        raise ContinuationError("preregistered continuation recipe drifted")
    if not (
        step50["training_checkpoint_marker_id"] == SOURCE_MARKER_ID
        and step50["training_checkpoint_marker_sha256"] == SOURCE_MARKER_SHA256
        and step50["optimizer_step"] == 50
        and decision["same_call_same_representation"]["step50_macro_f1"] == 0.6693876444923534
        and decision["leakage_aware_threshold_diagnostics"]["five_fold_group_held_macro_f1"] == 0.876661184941553
        and decision["leakage_aware_threshold_diagnostics"]["leave_one_source_out_macro_f1"] == 0.8749996527768132
        and decision["decision"]["further_paid_training_authorized"] is False
    ):
        raise ContinuationError("measured step50/Confirmatory600 decision drifted")
    rows: dict[str, list[dict[str, Any]]] = {}
    for role, filename in PACK_FILES.items():
        path = PACK / filename
        expected = manifest["artifacts"][filename]
        if verify_artifacts and sha_file(path) != expected["sha256"]:
            raise ContinuationError(f"continuation artifact drifted: {role}")
        rows[role] = read_jsonl(path)
        if len(rows[role]) != int(expected["rows"]):
            raise ContinuationError(f"continuation row count drifted: {role}")
    ids = [str(row["example_id"]) for row in rows["train_examples"]]
    flattened = [str(item) for step in rows["optimizer_steps"] for item in step["example_ids"]]
    if not (
        len(ids) == 800
        and len(set(ids)) == 800
        and ids == [str(row["example_id"]) for row in rows["train_encoded"]]
        and ids == [str(row["example_id"]) for row in rows["loss_weight_spans"]]
        and ids == flattened
        and len(rows["optimizer_steps"]) == TOTAL_STEPS
        and len({str(row["parent_id"]) for row in rows["train_examples"]}) == 800
    ):
        raise ContinuationError("continuation order/unique-parent contract drifted")
    for offset in range(0, TOTAL_STEPS - (TOTAL_STEPS % 3), 3):
        counts = Counter(value for step in rows["optimizer_steps"][offset:offset + 3] for value in step["verdicts"])
        if counts != Counter({verdict: 8 for verdict in VERDICTS}):
            raise ContinuationError("three-step verdict balance drifted")
    panels: dict[str, list[dict[str, Any]]] = {}
    for role in PANEL_ROLES:
        bound = manifest["panel_bindings"][role]
        path = _local_artifact_path(bound)
        if verify_artifacts and sha_file(path) != bound["sha256"]:
            raise ContinuationError(f"locked panel drifted: {role}")
        panels[role] = read_jsonl(path)
        if len(panels[role]) != int(bound["rows"]):
            raise ContinuationError(f"locked panel row count drifted: {role}")
    train_ids = {str(row[field]) for row in rows["train_examples"] for field in ("example_id", "parent_id", "record_id")}
    panel_ids = {
        str(value)
        for role in PANEL_ROLES
        for row in panels[role]
        for value in (row.get("example_id"), row.get("question_group_id"), row.get("coverage", {}).get("parent_example_id"))
        if value is not None
    }
    if train_ids & panel_ids:
        raise ContinuationError("continuation parent overlaps a locked evaluation panel")
    return {"manifest": manifest, "spec": spec, "step50": step50, "decision": decision, "rows": rows, "panels": panels}


def continuation_causal_loss(logits: Any, labels: Any, verdict_spans: Sequence[tuple[int, int]]) -> Any:
    """Inference-aligned causal loss: 45% verdict value, 55% explanation."""
    import torch
    import torch.nn.functional as functional

    if len(verdict_spans) != labels.shape[0]:
        raise ContinuationError("verdict-span cardinality drifted")
    shifted_logits = logits[:, :-1, :]
    shifted_labels = labels[:, 1:]
    valid = shifted_labels.ne(-100)
    parent_losses = []
    for index in range(labels.shape[0]):
        selected_logits = shifted_logits[index, valid[index], :]
        selected_labels = shifted_labels[index, valid[index]]
        if selected_labels.numel() == 0:
            raise ContinuationError("parent has no supervised target")
        losses = functional.cross_entropy(selected_logits.float(), selected_labels, reduction="none")
        weights = torch.tensor(
            factorial.verdict_weight_vector(int(losses.shape[0]), *verdict_spans[index], verdict_mass=VERDICT_LOSS_MASS),
            dtype=losses.dtype,
            device=losses.device,
        )
        parent_losses.append((losses * weights).sum())
    return torch.stack(parent_losses).mean()


def _augment_evaluation(result: dict[str, Any], panels: Mapping[str, Sequence[Mapping[str, Any]]]) -> dict[str, Any]:
    calibrated = result["panels"]
    raw: dict[str, Any] = {}
    for role in PANEL_ROLES:
        score_rows = result["constrained_score_rows"][role]
        predicted = [factorial._validate_score_map(row["scores"]) for row in score_rows]
        calibrated[role]["predicted_class_counts"] = dict(sorted(Counter(predicted).items()))
        calibrated[role]["distinct_predicted_classes"] = len(set(predicted))
        calibrated[role]["invalid_score_rows"] = 0
        raw[role] = proven._metrics(panels[role], [row["scores"] for row in score_rows], 1.0)
        raw[role]["predicted_class_counts"] = dict(sorted(Counter(predicted).items()))
        raw[role]["distinct_predicted_classes"] = len(set(predicted))
        raw[role]["invalid_score_rows"] = 0
    result["calibrated_panels"] = calibrated
    result["raw_temperature1_panels"] = raw
    result.pop("panels", None)
    return result


def soft_regression_gate(baseline: Mapping[str, Any], running_best: Mapping[str, Any], candidate: Mapping[str, Any]) -> dict[str, Any]:
    """Flag any preregistered regression for the patience-two soft stop."""
    gates: dict[str, bool] = {}
    deltas: dict[str, float] = {}
    for representation in ("raw_temperature1_panels", "calibrated_panels"):
        for role in PANEL_ROLES:
            after = candidate[representation][role]
            for reference_name, reference in (("step0", baseline), ("running_best", running_best)):
                before = reference[representation][role]
                prefix = f"{reference_name}__{representation}__{role}"
                macro_delta = float(after["macro_f1"]) - float(before["macro_f1"])
                deltas[f"{prefix}__macro_f1"] = macro_delta
                gates[f"{prefix}__macro_f1"] = macro_delta >= -REGRESSION_TOLERANCE
                for label in VERDICTS:
                    class_delta = float(after["per_class_f1"][label]["f1"]) - float(before["per_class_f1"][label]["f1"])
                    deltas[f"{prefix}__{label}_f1"] = class_delta
                    gates[f"{prefix}__{label}_f1"] = class_delta >= -REGRESSION_TOLERANCE
                supported_recall_delta = float(after["per_class_f1"]["supported"]["recall"]) - float(before["per_class_f1"]["supported"]["recall"])
                deltas[f"{prefix}__supported_recall"] = supported_recall_delta
                gates[f"{prefix}__supported_recall"] = supported_recall_delta >= -REGRESSION_TOLERANCE
                ece_delta = float(after["expected_calibration_error"]) - float(before["expected_calibration_error"])
                brier_delta = float(after["brier_score"]) - float(before["brier_score"])
                deltas[f"{prefix}__ece"] = ece_delta
                deltas[f"{prefix}__brier"] = brier_delta
                gates[f"{prefix}__ece"] = ece_delta <= REGRESSION_TOLERANCE
                gates[f"{prefix}__brier"] = brier_delta <= REGRESSION_TOLERANCE
            gates[f"integrity__{representation}__{role}__invalid"] = int(after.get("invalid_score_rows", 0)) == 0
            gates[f"integrity__{representation}__{role}__class_diversity"] = int(after.get("distinct_predicted_classes", 0)) >= 2
    return {
        "status": "soft_no_regression_pass" if all(gates.values()) else "soft_regression_observed",
        "all_gates_passed": all(gates.values()),
        "gates": gates,
        "deltas": deltas,
        "failed_gates": sorted(key for key, passed in gates.items() if not passed),
    }


def catastrophic_regression_gate(baseline: Mapping[str, Any], candidate: Mapping[str, Any]) -> dict[str, Any]:
    """Immediate-stop conditions; baseline-zero classes are not false alarms."""
    gates: dict[str, bool] = {}
    observations: dict[str, float] = {}
    for representation in ("raw_temperature1_panels", "calibrated_panels"):
        for role in PANEL_ROLES:
            before = baseline[representation][role]
            after = candidate[representation][role]
            for label in VERDICTS:
                before_f1 = float(before["per_class_f1"][label]["f1"])
                after_f1 = float(after["per_class_f1"][label]["f1"])
                key = f"{representation}__{role}__{label}_remains_nonzero"
                gates[key] = not (before_f1 > 0.0 and after_f1 <= REGRESSION_TOLERANCE)
            recall_drop = float(before["per_class_f1"]["supported"]["recall"]) - float(after["per_class_f1"]["supported"]["recall"])
            observations[f"{representation}__{role}__supported_recall_drop"] = recall_drop
            gates[f"{representation}__{role}__supported_recall_not_catastrophic"] = recall_drop < SUPPORTED_RECALL_CATASTROPHIC_DROP - REGRESSION_TOLERANCE
            gates[f"{representation}__{role}__finite"] = all(
                math.isfinite(float(value))
                for value in (after["macro_f1"], after["expected_calibration_error"], after["brier_score"])
            )
            gates[f"{representation}__{role}__valid_scores"] = int(after.get("invalid_score_rows", 0)) == 0
    return {
        "status": "catastrophic_gate_pass" if all(gates.values()) else "catastrophic_regression_stop",
        "all_gates_passed": all(gates.values()),
        "gates": gates,
        "observations": observations,
        "failed_gates": sorted(key for key, passed in gates.items() if not passed),
    }


def best_rank(evaluation: Mapping[str, Any]) -> tuple[float, ...]:
    panels = evaluation["calibrated_panels"]
    group = panels["group_held_examples"]
    source = panels["source_held_examples"]
    calibration = panels["calibration_examples"]
    return (
        statistics.fmean([float(group["macro_f1"]), float(source["macro_f1"])]),
        min(float(group["macro_f1"]), float(source["macro_f1"])),
        statistics.fmean([float(group["per_class_f1"]["supported"]["recall"]), float(source["per_class_f1"]["supported"]["recall"])]),
        -float(calibration["expected_calibration_error"]),
        -float(calibration["brier_score"]),
        -float(evaluation["optimizer_step"]),
    )


def _next_step_fields(steps: Sequence[Mapping[str, Any]], cursor: int) -> dict[str, Any]:
    ids = [] if cursor >= TOTAL_STEPS else [str(value) for value in steps[cursor]["example_ids"]]
    parents = [] if cursor >= TOTAL_STEPS else [str(value) for value in steps[cursor]["parent_ids"]]
    return {
        "next_optimizer_step": cursor + 1,
        "next_eight_example_ids": ids,
        "next_eight_example_ids_sha256": sha_bytes(canonical(ids).encode("utf-8")),
        "next_eight_parent_ids": parents,
        "next_eight_parent_ids_sha256": sha_bytes(canonical(parents).encode("utf-8")),
    }


def _validate_workspace_receipt(value: Mapping[str, Any], now: float) -> None:
    if not (
        valid_contract(value, "receipt_id")
        and isinstance(value.get("modal_profile"), str) and value["modal_profile"]
        and isinstance(value.get("workspace_slug"), str) and value["workspace_slug"]
        and value.get("owned_workspace_authorized") is True
        and value.get("no_running_apps_observed") is True
        and value.get("no_active_containers_observed") is True
        and float(value.get("available_credit_usd", -1)) >= MINIMUM_OUTSIDE_RESERVE_USD + MAXIMUM_TOTAL_COMPUTE_CHARGE_USD
        and 0 <= now - float(value.get("captured_epoch", 0)) <= 900
    ):
        raise ContinuationError("fresh owned-workspace receipt is invalid")


def _launch_ast_is_detached(source: str) -> bool:
    tree = ast.parse(source)
    launch = next((node for node in tree.body if isinstance(node, ast.FunctionDef) and node.name == "launch_one"), None)
    if launch is None:
        return False
    spawns = [node for node in ast.walk(launch) if isinstance(node, ast.Call) and isinstance(node.func, ast.Attribute) and node.func.attr == "spawn"]
    waits = [
        node for node in ast.walk(launch)
        if isinstance(node, ast.Call)
        and isinstance(node.func, ast.Attribute)
        and (
            node.func.attr == "remote"
            or (node.func.attr == "get" and isinstance(node.func.value, ast.Name) and node.func.value.id == "call")
        )
    ]
    return len(spawns) == 1 and not waits


def _image_excludes_private_label_artifacts(source: str) -> bool:
    tree = ast.parse(source)
    builder = next((node for node in tree.body if isinstance(node, ast.FunctionDef) and node.name == "_build_image"), None)
    if builder is None:
        return False
    constants = {node.value.casefold() for node in ast.walk(builder) if isinstance(node, ast.Constant) and isinstance(node.value, str)}
    return not any("private_expectation" in value or "confirmatory600" in value for value in constants)


def _authorization_before_model_load_ast(source: str) -> bool:
    tree = ast.parse(source)
    remote = next((node for node in tree.body if isinstance(node, ast.FunctionDef) and node.name == "_run_remote"), None)
    if remote is None:
        return False
    consume_lines = [node.lineno for node in ast.walk(remote) if isinstance(node, ast.Call) and isinstance(node.func, ast.Name) and node.func.id == "_consume_authorization"]
    load_lines = [node.lineno for node in ast.walk(remote) if isinstance(node, ast.Call) and isinstance(node.func, ast.Attribute) and node.func.attr == "from_pretrained" and isinstance(node.func.value, ast.Name) and node.func.value.id == "AutoModelForMultimodalLM"]
    return len(consume_lines) == 1 and len(load_lines) == 1 and consume_lines[0] < load_lines[0]


def static_acceptance() -> dict[str, Any]:
    bundle = _load_pack(verify_artifacts=True)
    source = Path(__file__).read_text(encoding="utf-8")
    spec = bundle["spec"]
    verdicts = Counter(row["verdict"] for row in bundle["rows"]["train_examples"])
    checks = {
        "sealed_pack_and_exact_step50_evidence": True,
        "single_lr2e6_recipe_only": spec["single_preregistered_recipe"]["learning_rate"] == 2e-6,
        "verdict45_explanation55_no_auxiliary": spec["single_preregistered_recipe"]["verdict_value_loss_mass"] == 0.45 and spec["single_preregistered_recipe"]["auxiliary_objective"] is None,
        "exact_800_unique_parent_rows": len(bundle["rows"]["train_examples"]) == 800 and len({row["parent_id"] for row in bundle["rows"]["train_examples"]}) == 800,
        "near_exact_global_three_way_balance": max(verdicts.values()) - min(verdicts.values()) == 1,
        "exact_each_complete_three_step_balance": spec["curriculum"]["each_complete_three_step_block_verdict_counts"] == {verdict: 8 for verdict in VERDICTS},
        "hard_supported_families_and_nei_counterexamples_present": spec["curriculum"]["hard_positive_tag_counts"]["preregistered_hard_family"] >= 90 and verdicts["not_enough_information"] >= 106,
        "no_train_eval_parent_overlap": True,
        "fresh_optimizer_old_state_forbidden": spec["initialization"]["fresh_optimizer_required"] is True and spec["initialization"]["old_optimizer_scheduler_rng_forbidden"] is True,
        "stage_a60_optional_stage_b40": STAGE_A_END == 60 and STAGE_B_END == 100,
        "eval_every10_through100": EVALUATION_STEPS == tuple(range(0, 101, 10)),
        "raw_calibrated_class_recall_soft_and_catastrophic_gates": all(token in source for token in ("raw_temperature1_panels", "supported_recall", "soft_regression_gate", "catastrophic_regression_gate", "running_best")),
        "patience_two_for_modest_dips": SOFT_REGRESSION_PATIENCE == 2 and "consecutive_soft_regressions" in source,
        "separate_stage_authorization": "stage_id" in source and "stage_b_eligibility" in source,
        "best_checkpoint_separate_from_last": "best-checkpoint.json" in source and "best_rank" in source,
        "atomic_checkpoint_publication": "os.replace(staging, final)" in source,
        "source_marker_tree_adapter_all_exact": all(token in source for token in (SOURCE_MARKER_SHA256, SOURCE_CHECKPOINT_TREE_SHA256, SOURCE_ADAPTER_TREE_SHA256)),
        "private_confirmatory_labels_not_in_pack_or_image": bundle["manifest"]["private_confirmatory600_labels_included"] is False and _image_excludes_private_label_artifacts(source),
        "network_blocked_h200_one_container_zero_retry": all(token in source for token in ('gpu="H200"', "block_network=True", "max_containers=1", "retries=0")),
        "exact_cost_cap_and_reserves": MAXIMUM_TOTAL_COMPUTE_CHARGE_USD == 1.25 and MAXIMUM_GPU_CHARGE_USD == 1.05 and PERSISTENCE_RESERVE_USD == 0.12 and MINIMUM_OUTSIDE_RESERVE_USD == 10.0,
        "one_use_authorization_consumed_before_model_load": _authorization_before_model_load_ast(source),
        "detached_single_spawn_no_tether": _launch_ast_is_detached(source),
        "launch_not_authorized_by_acceptance": True,
    }
    if not all(checks.values()):
        raise ContinuationError(f"static acceptance failed: {[key for key, passed in checks.items() if not passed]}")
    return content_contract(
        {
            "version": ACCEPTANCE_VERSION,
            "status": "ALL_STATIC_ACCEPTANCE_CHECKS_PASS_BUILD_ONLY_NO_LAUNCH_AUTHORITY",
            "executor_sha256": sha_file(Path(__file__)),
            "manifest_sha256": MANIFEST_SHA256,
            "manifest_id": MANIFEST_ID,
            "spec_sha256": SPEC_SHA256,
            "spec_id": SPEC_ID,
            "checks": checks,
            "confirmatory600_observed": {"direct_macro_f1": 0.6693876444923534, "group_held_calibrated_macro_f1": 0.876661184941553, "source_held_calibrated_macro_f1": 0.8749996527768132},
            "authorization_present": False,
            "launch_performed": False,
            "gpu_requested": False,
            "model_weights_loaded": False,
            "optimizer_created": False,
            "network_action_performed": False,
            "paid_action_performed": False,
            "prohibited_runtime_tests_run": False,
        },
        "acceptance_id",
    )


def write_static_acceptance(path: Path = DEFAULT_ACCEPTANCE) -> dict[str, Any]:
    receipt = static_acceptance()
    atomic_json(path, receipt)
    return receipt


def _validate_acceptance(value: Mapping[str, Any], executor_sha: str | None = None) -> dict[str, Any]:
    if not (
        valid_contract(value, "acceptance_id")
        and value.get("version") == ACCEPTANCE_VERSION
        and value.get("status") == "ALL_STATIC_ACCEPTANCE_CHECKS_PASS_BUILD_ONLY_NO_LAUNCH_AUTHORITY"
        and value.get("executor_sha256") == (executor_sha or sha_file(Path(__file__)))
        and value.get("manifest_sha256") == MANIFEST_SHA256
        and value.get("manifest_id") == MANIFEST_ID
        and value.get("spec_sha256") == SPEC_SHA256
        and value.get("spec_id") == SPEC_ID
        and value.get("checks") and all(value["checks"].values())
        and value.get("paid_action_performed") is False
    ):
        raise ContinuationError("static acceptance is invalid")
    return dict(value)


def emit_safe_commands(acceptance_path: Path, output: Path = DEFAULT_COMMANDS) -> dict[str, Any]:
    acceptance = _validate_acceptance(read_json(acceptance_path))
    module = "pipeline/modal_train_gemma4_e4b_step50_guarded_continuation_v1.py"
    value = content_contract(
        {
            "version": VERSION,
            "status": "SAFE_TEMPLATES_ONLY_NO_AUTHORIZATION_NO_LAUNCH",
            "acceptance_id": acceptance["acceptance_id"],
            "dry_plan": f"python -m pipeline.modal_train_gemma4_e4b_step50_guarded_continuation_v1 --plan --acceptance {acceptance_path}",
            "stage_a_mint_template": f"python -m pipeline.modal_train_gemma4_e4b_step50_guarded_continuation_v1 --mint-authorization --acceptance {acceptance_path} --workspace-receipt <fresh.json> --run-id <run_id> --stage-id stage_a --output-authorization <authorization.json>",
            "stage_b_mint_template": f"python -m pipeline.modal_train_gemma4_e4b_step50_guarded_continuation_v1 --mint-authorization --acceptance {acceptance_path} --workspace-receipt <fresh.json> --run-id <same_run_id> --stage-id stage_b --stage-a-checkpoint <downloaded_step60_complete.json> --stage-a-eligibility <downloaded_stage-b-eligibility.json> --output-authorization <authorization.json>",
            "detached_launch_template": f"$env:MODAL_PROFILE='<bound_profile>'; $env:BICHAR_MODAL_DETACHED='1'; modal run --detach {module}::launch_one --run-id <run_id> --stage-id <stage_a_or_stage_b> --authorization-path <authorization.json> --detached-attestation",
            "maximum_total_compute_charge_usd": MAXIMUM_TOTAL_COMPUTE_CHARGE_USD,
            "minimum_outside_credit_reserve_usd": MINIMUM_OUTSIDE_RESERVE_USD,
            "paid_action_performed": False,
        },
        "commands_id",
    )
    atomic_json(output, value)
    return value


def mint_authorization(
    *,
    acceptance_path: Path,
    workspace_receipt_path: Path,
    run_id: str,
    stage_id: str,
    stage_a_checkpoint_path: Path | None = None,
    stage_a_eligibility_path: Path | None = None,
) -> dict[str, Any]:
    _load_pack(verify_artifacts=True)
    acceptance = _validate_acceptance(read_json(acceptance_path))
    workspace = read_json(workspace_receipt_path)
    now = time.time()
    _validate_workspace_receipt(workspace, now)
    if not _valid_run_id(run_id) or stage_id not in STAGE_IDS:
        raise ContinuationError("invalid run or stage ID")
    if stage_id == "stage_a":
        if stage_a_checkpoint_path is not None or stage_a_eligibility_path is not None:
            raise ContinuationError("stage A authorization cannot bind stage A outputs")
        stage_source = {
            "mode": "immutable_step50_adapter_fresh_optimizer",
            "start_optimizer_step": 0,
            "end_optimizer_step": STAGE_A_END,
            "checkpoint_marker_id": SOURCE_MARKER_ID,
            "checkpoint_marker_sha256": SOURCE_MARKER_SHA256,
            "adapter_tree_sha256": SOURCE_ADAPTER_TREE_SHA256,
        }
    else:
        if stage_a_checkpoint_path is None or stage_a_eligibility_path is None:
            raise ContinuationError("stage B requires downloaded stage-A checkpoint and eligibility receipts")
        marker = read_json(stage_a_checkpoint_path)
        eligibility = read_json(stage_a_eligibility_path)
        if not (
            valid_contract(marker, "marker_id")
            and marker.get("status") == "atomic_step50_continuation_checkpoint_complete"
            and marker.get("run_id") == run_id
            and marker.get("manifest_id") == MANIFEST_ID
            and marker.get("spec_id") == SPEC_ID
            and marker.get("optimizer_step") == STAGE_A_END
            and valid_contract(eligibility, "eligibility_id")
            and eligibility.get("status") == "stage_b_eligible_stable_or_improving_boundary"
            and eligibility.get("run_id") == run_id
            and eligibility.get("stage_a_optimizer_step") == STAGE_A_END
            and eligibility.get("stage_a_checkpoint_marker_id") == marker["marker_id"]
            and eligibility.get("stage_a_checkpoint_marker_sha256") == sha_file(stage_a_checkpoint_path)
            and eligibility.get("stage_a_checkpoint_tree_sha256") == marker["checkpoint_tree_sha256"]
            and eligibility.get("stage_a_adapter_tree_sha256") == marker["adapter_tree_sha256"]
            and re.fullmatch(r"[0-9a-f]{64}", str(eligibility.get("stage_a_trajectory_id", "")))
            and re.fullmatch(r"[0-9a-f]{64}", str(eligibility.get("stage_a_trajectory_sha256", "")))
            and eligibility.get("stage_b_authorization_allowed") is True
            and eligibility.get("promotion_allowed") is False
        ):
            raise ContinuationError("stage-A boundary is not eligible for stage B")
        stage_source = {
            "mode": "resume_stage_a_step60_optimizer_and_adapter",
            "start_optimizer_step": STAGE_A_END,
            "end_optimizer_step": STAGE_B_END,
            "checkpoint_marker_id": marker["marker_id"],
            "checkpoint_marker_sha256": sha_file(stage_a_checkpoint_path),
            "checkpoint_tree_sha256": marker["checkpoint_tree_sha256"],
            "adapter_tree_sha256": marker["adapter_tree_sha256"],
            "stage_b_eligibility_id": eligibility["eligibility_id"],
            "stage_b_eligibility_sha256": sha_file(stage_a_eligibility_path),
            "stage_a_trajectory_id": eligibility["stage_a_trajectory_id"],
            "stage_a_trajectory_sha256": eligibility["stage_a_trajectory_sha256"],
        }
    authorization = content_contract(
        {
            "version": AUTH_VERSION,
            "status": "fresh_one_use_step50_continuation_authorization",
            "executor_sha256": sha_file(Path(__file__)),
            "manifest_sha256": MANIFEST_SHA256,
            "manifest_id": MANIFEST_ID,
            "spec_sha256": SPEC_SHA256,
            "spec_id": SPEC_ID,
            "acceptance_receipt": acceptance,
            "workspace_receipt": workspace,
            "run_id": run_id,
            "stage_id": stage_id,
            "stage_source": stage_source,
            "source_marker_id": SOURCE_MARKER_ID,
            "source_marker_sha256": SOURCE_MARKER_SHA256,
            "source_checkpoint_tree_sha256": SOURCE_CHECKPOINT_TREE_SHA256,
            "source_adapter_tree_sha256": SOURCE_ADAPTER_TREE_SHA256,
            "learning_rate": LEARNING_RATE,
            "maximum_steps": TOTAL_STEPS,
            "evaluation_steps": list(EVALUATION_STEPS),
            "issued_epoch": now,
            "expires_epoch": now + 900,
            "authorize_exactly_one_detached_h200_call": True,
            "fresh_optimizer_required": True,
            "private_competition_labels_allowed": False,
            "network_blocked_required": True,
            "single_use_container_required": True,
            "maximum_containers": 1,
            "retries": 0,
            "maximum_total_compute_charge_usd": MAXIMUM_TOTAL_COMPUTE_CHARGE_USD,
            "maximum_gpu_charge_usd": MAXIMUM_GPU_CHARGE_USD,
            "persistence_reserve_usd": PERSISTENCE_RESERVE_USD,
            "minimum_outside_credit_reserve_usd": MINIMUM_OUTSIDE_RESERVE_USD,
            "promotion_allowed": False,
        },
        "authorization_id",
    )
    return authorization


def _validate_authorization(value: Mapping[str, Any], *, run_id: str, stage_id: str, executor_sha: str, now: float) -> dict[str, Any]:
    acceptance = value.get("acceptance_receipt")
    workspace = value.get("workspace_receipt")
    if not (
        valid_contract(value, "authorization_id")
        and value.get("version") == AUTH_VERSION
        and value.get("status") == "fresh_one_use_step50_continuation_authorization"
        and value.get("executor_sha256") == executor_sha
        and value.get("manifest_sha256") == MANIFEST_SHA256
        and value.get("manifest_id") == MANIFEST_ID
        and value.get("spec_sha256") == SPEC_SHA256
        and value.get("spec_id") == SPEC_ID
        and value.get("run_id") == run_id and _valid_run_id(run_id)
        and value.get("stage_id") == stage_id and stage_id in STAGE_IDS
        and isinstance(value.get("stage_source"), Mapping)
        and value.get("source_marker_id") == SOURCE_MARKER_ID
        and value.get("source_marker_sha256") == SOURCE_MARKER_SHA256
        and value.get("source_checkpoint_tree_sha256") == SOURCE_CHECKPOINT_TREE_SHA256
        and value.get("source_adapter_tree_sha256") == SOURCE_ADAPTER_TREE_SHA256
        and value.get("learning_rate") == LEARNING_RATE
        and value.get("maximum_steps") == TOTAL_STEPS
        and value.get("evaluation_steps") == list(EVALUATION_STEPS)
        and value.get("authorize_exactly_one_detached_h200_call") is True
        and value.get("fresh_optimizer_required") is True
        and value.get("private_competition_labels_allowed") is False
        and value.get("network_blocked_required") is True
        and value.get("single_use_container_required") is True
        and value.get("maximum_containers") == 1 and value.get("retries") == 0
        and value.get("maximum_total_compute_charge_usd") == MAXIMUM_TOTAL_COMPUTE_CHARGE_USD
        and value.get("maximum_gpu_charge_usd") == MAXIMUM_GPU_CHARGE_USD
        and value.get("persistence_reserve_usd") == PERSISTENCE_RESERVE_USD
        and value.get("minimum_outside_credit_reserve_usd") == MINIMUM_OUTSIDE_RESERVE_USD
        and value.get("promotion_allowed") is False
        and float(value.get("issued_epoch", math.inf)) <= now <= float(value.get("expires_epoch", -math.inf))
        and isinstance(acceptance, Mapping) and isinstance(workspace, Mapping)
    ):
        raise ContinuationError("one-use continuation authorization is invalid")
    _validate_acceptance(acceptance, executor_sha)
    _validate_workspace_receipt(workspace, now)
    stage_source = value["stage_source"]
    if stage_id == "stage_a":
        expected = {
            "mode": "immutable_step50_adapter_fresh_optimizer",
            "start_optimizer_step": 0,
            "end_optimizer_step": STAGE_A_END,
            "checkpoint_marker_id": SOURCE_MARKER_ID,
            "checkpoint_marker_sha256": SOURCE_MARKER_SHA256,
            "adapter_tree_sha256": SOURCE_ADAPTER_TREE_SHA256,
        }
        if stage_source != expected:
            raise ContinuationError("stage A source binding drifted")
    else:
        required_hashes = ("checkpoint_marker_id", "checkpoint_marker_sha256", "checkpoint_tree_sha256", "adapter_tree_sha256", "stage_b_eligibility_id", "stage_b_eligibility_sha256", "stage_a_trajectory_id", "stage_a_trajectory_sha256")
        if not (
            stage_source.get("mode") == "resume_stage_a_step60_optimizer_and_adapter"
            and stage_source.get("start_optimizer_step") == STAGE_A_END
            and stage_source.get("end_optimizer_step") == STAGE_B_END
            and all(re.fullmatch(r"[0-9a-f]{64}", str(stage_source.get(field, ""))) for field in required_hashes)
        ):
            raise ContinuationError("stage B source binding drifted")
    return dict(value)


def build_local_plan(acceptance_path: Path | None = None, authorization_path: Path | None = None) -> dict[str, Any]:
    blockers: list[str] = []
    authorization: dict[str, Any] | None = None
    try:
        _load_pack(verify_artifacts=True)
    except Exception as exc:
        blockers.append(f"sealed_pack_invalid:{type(exc).__name__}")
    if acceptance_path is None:
        blockers.append("static_acceptance_receipt_missing")
    else:
        try:
            _validate_acceptance(read_json(acceptance_path))
        except Exception as exc:
            blockers.append(f"static_acceptance_invalid:{type(exc).__name__}")
    if authorization_path is None:
        blockers.append("fresh_one_use_authorization_missing")
    else:
        try:
            candidate = read_json(authorization_path)
            authorization = _validate_authorization(candidate, run_id=str(candidate.get("run_id")), stage_id=str(candidate.get("stage_id")), executor_sha=sha_file(Path(__file__)), now=time.time())
            if os.environ.get("MODAL_PROFILE") != authorization["workspace_receipt"]["modal_profile"]:
                raise ContinuationError("MODAL_PROFILE differs from bound receipt")
        except Exception as exc:
            blockers.append(f"authorization_invalid:{type(exc).__name__}")
    return {
        "version": VERSION,
        "status": "READY_TO_SPAWN_EXACTLY_ONE_DETACHED_CONTINUATION" if not blockers else "NO_GO",
        "launch_ready": not blockers,
        "blockers": blockers,
        "run_id": authorization.get("run_id") if authorization else None,
        "stage_id": authorization.get("stage_id") if authorization else None,
        "manifest_id": MANIFEST_ID,
        "spec_id": SPEC_ID,
        "executor_sha256": sha_file(Path(__file__)),
        "maximum_steps": TOTAL_STEPS,
        "stage_a_steps": STAGE_A_END,
        "stage_b_optional_steps": STAGE_B_END - STAGE_A_END,
        "evaluation_steps": list(EVALUATION_STEPS),
        "maximum_total_compute_charge_usd": MAXIMUM_TOTAL_COMPUTE_CHARGE_USD,
        "maximum_gpu_charge_usd": MAXIMUM_GPU_CHARGE_USD,
        "persistence_reserve_usd": PERSISTENCE_RESERVE_USD,
        "minimum_outside_credit_reserve_usd": MINIMUM_OUTSIDE_RESERVE_USD,
        "launch_performed": False,
        "paid_action_performed": False,
    }


def _build_image() -> modal.Image:
    bundle = _load_pack(verify_artifacts=True)
    image = (
        modal.Image.debian_slim(python_version="3.11")
        .pip_install(*factorial.FACTORIAL_ENVIRONMENT_PACKAGES)
        .env({"PYTHONPATH": "/root", "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True", "PYTORCH_ALLOC_CONF": "expandable_segments:True"})
        .add_local_file(str(Path(__file__)), remote_path="/contracts/continuation_executor.py")
        .add_local_file(str(Path(__file__)), remote_path="/root/pipeline/modal_train_gemma4_e4b_step50_guarded_continuation_v1.py")
        .add_local_file(str(Path(factorial.__file__)), remote_path="/root/pipeline/modal_train_gemma4_e4b_contextual_repair_factorial_canary_v1.py")
        .add_local_file(str(Path(proven.__file__)), remote_path="/root/pipeline/modal_train_gemma4_e4b_contextual_heavy14m_v1.py")
        .add_local_file(str(MANIFEST), remote_path="/contracts/manifest.json")
        .add_local_file(str(SPEC), remote_path="/contracts/continuation_spec.json")
        .add_local_file(str(factorial.MODEL_MANIFEST), remote_path="/contracts/model_manifest.json")
        .add_local_file(str(factorial.STAGE_RECEIPT), remote_path="/contracts/stage_receipt.json")
        .add_local_file(str(factorial.TOKENIZER_RECEIPT), remote_path="/contracts/tokenizer_receipt.json")
    )
    for role, filename in PACK_FILES.items():
        image = image.add_local_file(str(PACK / filename), remote_path=f"/continuation/{role}.jsonl")
    for role in PANEL_ROLES:
        image = image.add_local_file(str(_local_artifact_path(bundle["manifest"]["panel_bindings"][role])), remote_path=f"/panels/{role}.jsonl")
    return image


def _select_image() -> modal.Image:
    return _build_image() if modal.is_local() else modal.Image.debian_slim(python_version="3.11")


def _remote_bundle() -> dict[str, Any]:
    manifest = read_json(Path("/contracts/manifest.json"), MANIFEST_SHA256)
    spec = read_json(Path("/contracts/continuation_spec.json"), SPEC_SHA256)
    if not (valid_contract(manifest, "manifest_id") and manifest["manifest_id"] == MANIFEST_ID and valid_contract(spec, "spec_id") and spec["spec_id"] == SPEC_ID):
        raise ContinuationError("remote manifest/spec drifted")
    rows: dict[str, list[dict[str, Any]]] = {}
    for role, filename in PACK_FILES.items():
        path = Path("/continuation") / f"{role}.jsonl"
        expected = manifest["artifacts"][filename]
        if sha_file(path) != expected["sha256"]:
            raise ContinuationError(f"remote continuation artifact drifted: {role}")
        rows[role] = read_jsonl(path)
    panels: dict[str, list[dict[str, Any]]] = {}
    for role in PANEL_ROLES:
        path = Path("/panels") / f"{role}.jsonl"
        expected = manifest["panel_bindings"][role]
        if sha_file(path) != expected["sha256"]:
            raise ContinuationError(f"remote locked panel drifted: {role}")
        panels[role] = read_jsonl(path)
    ids = [str(row["example_id"]) for row in rows["train_examples"]]
    if not (len(ids) == 800 and ids == [str(row["example_id"]) for row in rows["train_encoded"]] == [str(row["example_id"]) for row in rows["loss_weight_spans"]] == [str(item) for step in rows["optimizer_steps"] for item in step["example_ids"]]):
        raise ContinuationError("remote exact training order drifted")
    return {"manifest": manifest, "spec": spec, "rows": rows, "panels": panels}


def _verify_source_step50() -> Path:
    root = Path("/step50_source").resolve()
    checkpoint = (root / SOURCE_STEP50_RELATIVE).resolve()
    if root not in checkpoint.parents:
        raise ContinuationError("source checkpoint escapes read-only volume")
    marker = read_json(checkpoint / "complete.json", SOURCE_MARKER_SHA256)
    if not (
        valid_contract(marker, "marker_id")
        and marker["marker_id"] == SOURCE_MARKER_ID
        and marker.get("status") == "atomic_factorial_arm_checkpoint_complete"
        and marker.get("arm_id") == "lr1e5_verdict25"
        and marker.get("run_id") == SOURCE_RUN_ID
        and marker.get("optimizer_step") == 50
        and marker.get("processed_native_tokens") == 172716
        and marker.get("checkpoint_tree_sha256") == SOURCE_CHECKPOINT_TREE_SHA256
        and marker.get("adapter_tree_sha256") == SOURCE_ADAPTER_TREE_SHA256
        and marker.get("promotion_allowed") is False
    ):
        raise ContinuationError("immutable source step50 marker drifted")
    if tree_binding(checkpoint, exclude=frozenset({"complete.json"}))["tree_sha256"] != SOURCE_CHECKPOINT_TREE_SHA256:
        raise ContinuationError("immutable source checkpoint tree drifted")
    adapter = checkpoint / "adapter"
    if tree_binding(adapter)["tree_sha256"] != SOURCE_ADAPTER_TREE_SHA256:
        raise ContinuationError("immutable source adapter tree drifted")
    return adapter


def _consume_authorization(authorization: Mapping[str, Any], run_id: str, stage_id: str) -> None:
    root = Path("/continuation_output/authorizations") / str(authorization["authorization_id"])
    try:
        root.mkdir(parents=True, exist_ok=False)
    except FileExistsError as exc:
        raise ContinuationError("authorization already consumed") from exc
    atomic_json(root / "consumed.json", content_contract({"version": VERSION, "status": "consumed_before_model_load", "authorization_id": authorization["authorization_id"], "run_id": run_id, "stage_id": stage_id, "consumed_epoch": time.time()}, "consumption_id"))
    output_volume.commit()


def _bounded_cost(started: float) -> tuple[float, float, float]:
    seconds = time.perf_counter() - started + 30.0
    return seconds * TOTAL_RATE_USD_PER_SECOND, seconds * GPU_RATE_USD_PER_SECOND, seconds


def _charge_guard(started: float, deadline: float, recent_steps: Sequence[float], evaluation_due: bool) -> None:
    predicted = max(recent_steps[-3:], default=25.0) * 1.5
    evaluation = EVALUATION_RESERVE_SECONDS if evaluation_due else 0.0
    persistence = PERSISTENCE_RESERVE_USD / TOTAL_RATE_USD_PER_SECOND
    if time.perf_counter() + predicted + evaluation + persistence >= deadline:
        raise CleanBudgetStop("charge reserve reached before next atomic unit")
    total, gpu, _ = _bounded_cost(started)
    if total >= MAXIMUM_TOTAL_COMPUTE_CHARGE_USD - PERSISTENCE_RESERVE_USD or gpu >= MAXIMUM_GPU_CHARGE_USD:
        raise CleanBudgetStop("exact continuation cost cap reached")


def _verify_checkpoint(path: Path, bundle: Mapping[str, Any], run_id: str, executor_sha: str) -> dict[str, Any]:
    marker = read_json(path / "complete.json")
    cursor = int(marker.get("optimizer_step", -1))
    if not (
        valid_contract(marker, "marker_id")
        and marker.get("status") == "atomic_step50_continuation_checkpoint_complete"
        and marker.get("run_id") == run_id
        and marker.get("manifest_id") == MANIFEST_ID
        and marker.get("spec_id") == SPEC_ID
        and marker.get("executor_sha256") == executor_sha
        and marker.get("source_marker_id") == SOURCE_MARKER_ID
        and 0 <= cursor <= TOTAL_STEPS
        and all(marker.get(key) == value for key, value in _next_step_fields(bundle["rows"]["optimizer_steps"], cursor).items())
    ):
        raise ContinuationError("continuation checkpoint marker drifted")
    tree = tree_binding(path, exclude=frozenset({"complete.json"}))
    if marker.get("checkpoint_tree_sha256") != tree["tree_sha256"] or marker.get("adapter_tree_sha256") != tree_binding(path / "adapter")["tree_sha256"]:
        raise ContinuationError("continuation checkpoint tree drifted")
    return marker


def _publish_checkpoint(*, model: Any, optimizer: Any, scheduler_state: Mapping[str, Any], run_root: Path, bundle: Mapping[str, Any], run_id: str, cursor: int, processed_tokens: int, ledgers: Mapping[str, Any], started: float, kind: str) -> tuple[Path, dict[str, Any]]:
    import torch

    executor_sha = sha_file(Path("/contracts/continuation_executor.py"))
    final = run_root / "checkpoints" / f"step-{cursor:07d}"
    if final.exists():
        return final, _verify_checkpoint(final, bundle, run_id, executor_sha)
    staging = final.parent / f".step-{cursor:07d}.staging-{uuid.uuid4().hex}"
    staging.mkdir(parents=True, exist_ok=False)
    try:
        adapter = staging / "adapter"
        model.save_pretrained(adapter, safe_serialization=True)
        torch.save({"optimizer": optimizer.state_dict(), "scheduler": dict(scheduler_state), "python_rng": random.getstate(), "torch_rng": torch.get_rng_state(), "cuda_rng": torch.cuda.get_rng_state_all(), "sampler_cursor": cursor, "processed_native_tokens": processed_tokens}, staging / "fresh_optimizer_scheduler_rng.pt")
        atomic_json(staging / "ledgers.json", dict(ledgers))
        total, gpu, _ = _bounded_cost(started)
        if total > MAXIMUM_TOTAL_COMPUTE_CHARGE_USD or gpu > MAXIMUM_GPU_CHARGE_USD:
            raise ContinuationError("checkpoint exceeds exact cost cap")
        tree = tree_binding(staging, exclude=frozenset({"complete.json"}))
        marker = content_contract(
            {
                "version": VERSION,
                "status": "atomic_step50_continuation_checkpoint_complete",
                "run_id": run_id,
                "manifest_id": MANIFEST_ID,
                "spec_id": SPEC_ID,
                "executor_sha256": executor_sha,
                "source_marker_id": SOURCE_MARKER_ID,
                "source_adapter_tree_sha256": SOURCE_ADAPTER_TREE_SHA256,
                "optimizer_initialization": "fresh_at_continuation_step0",
                "optimizer_step": cursor,
                "processed_native_tokens": processed_tokens,
                **_next_step_fields(bundle["rows"]["optimizer_steps"], cursor),
                "adapter_tree_sha256": tree_binding(adapter)["tree_sha256"],
                "checkpoint_tree_sha256": tree["tree_sha256"],
                "checkpoint_kind": kind,
                "bounded_total_compute_cost_usd": total,
                "bounded_gpu_cost_usd": gpu,
                "promotion_allowed": False,
            },
            "marker_id",
        )
        atomic_json(staging / "complete.json", marker)
        final.parent.mkdir(parents=True, exist_ok=True)
        os.replace(staging, final)
        output_volume.commit()
        return final, _verify_checkpoint(final, bundle, run_id, executor_sha)
    except BaseException:
        shutil.rmtree(staging, ignore_errors=True)
        raise


def _publish_evaluation(*, model: Any, tokenizer: Any, bundle: Mapping[str, Any], run_root: Path, run_id: str, step: int, checkpoint_path: Path, checkpoint_marker: Mapping[str, Any], deadline: float, started: float) -> dict[str, Any]:
    final = run_root / "evaluations" / f"step-{step:07d}"
    if final.exists():
        summary = read_json(final / "summary.json")
        if not (valid_contract(summary, "evaluation_id") and summary.get("run_id") == run_id and summary.get("optimizer_step") == step):
            raise ContinuationError("existing evaluation drifted")
        return summary
    staging = final.parent / f".step-{step:07d}.staging-{uuid.uuid4().hex}"
    staging.mkdir(parents=True, exist_ok=False)
    try:
        result = _augment_evaluation(proven._score_milestone(model, tokenizer, bundle["panels"], step, deadline), bundle["panels"])
        ledgers = {role: write_jsonl(staging / f"{role}.jsonl", result["constrained_score_rows"][role]) for role in PANEL_ROLES}
        total, gpu, _ = _bounded_cost(started)
        summary = content_contract(
            {
                "version": VERSION,
                "status": "locked_continuation_evaluation_complete",
                "run_id": run_id,
                "manifest_id": MANIFEST_ID,
                "spec_id": SPEC_ID,
                "optimizer_step": step,
                "training_checkpoint_marker_id": checkpoint_marker["marker_id"],
                "training_checkpoint_marker_sha256": sha_file(checkpoint_path / "complete.json"),
                "temperature": result["temperature"],
                "temperature_fit_panel": "calibration_examples_only",
                "calibrated_panels": result["calibrated_panels"],
                "raw_temperature1_panels": result["raw_temperature1_panels"],
                "constrained_score_ledger_bindings": ledgers,
                "runtime_seconds": result["runtime_seconds"],
                "bounded_total_compute_cost_usd": total,
                "bounded_gpu_cost_usd": gpu,
                "private_confirmatory600_labels_uploaded": False,
                "promotion_allowed": False,
            },
            "evaluation_id",
        )
        atomic_json(staging / "summary.json", summary)
        tree = tree_binding(staging, exclude=frozenset({"complete.json"}))
        atomic_json(staging / "complete.json", content_contract({"version": VERSION, "status": "atomic_continuation_evaluation_complete", "run_id": run_id, "optimizer_step": step, "evaluation_id": summary["evaluation_id"], "evaluation_tree_sha256": tree["tree_sha256"]}, "marker_id"))
        final.parent.mkdir(parents=True, exist_ok=True)
        os.replace(staging, final)
        output_volume.commit()
        return summary
    except BaseException:
        shutil.rmtree(staging, ignore_errors=True)
        raise


def _publish_best(run_root: Path, step: int, evaluation: Mapping[str, Any], checkpoint_path: Path, checkpoint_marker: Mapping[str, Any]) -> dict[str, Any]:
    best = content_contract(
        {
            "version": VERSION,
            "status": "authenticated_best_not_last_checkpoint",
            "optimizer_step": step,
            "evaluation_id": evaluation["evaluation_id"],
            "evaluation_sha256": sha_file(run_root / "evaluations" / f"step-{step:07d}" / "summary.json"),
            "checkpoint_marker_id": checkpoint_marker["marker_id"],
            "checkpoint_marker_sha256": sha_file(checkpoint_path / "complete.json"),
            "checkpoint_tree_sha256": checkpoint_marker["checkpoint_tree_sha256"],
            "adapter_tree_sha256": checkpoint_marker["adapter_tree_sha256"],
            "rank": list(best_rank(evaluation)),
            "promotion_allowed": False,
        },
        "best_id",
    )
    atomic_json(run_root / "best-checkpoint.json", best)
    output_volume.commit()
    return best


def _write_terminal(run_root: Path, body: Mapping[str, Any], stage_id: str) -> dict[str, Any]:
    value = content_contract(dict(body) | {"version": VERSION, "promotion_allowed": False}, "terminal_id")
    atomic_json(run_root / f"terminal-{stage_id}.json", value)
    atomic_json(run_root / "terminal.json", value)
    output_volume.commit()
    return value


def _run_remote(run_id: str, stage_id: str, authorization: Mapping[str, Any]) -> dict[str, Any]:
    import torch
    from peft import PeftModel, prepare_model_for_kbit_training
    from transformers import AutoModelForMultimodalLM, AutoProcessor, BitsAndBytesConfig

    started = time.perf_counter()
    bundle = _remote_bundle()
    executor_sha = sha_file(Path("/contracts/continuation_executor.py"))
    authorization = _validate_authorization(authorization, run_id=run_id, stage_id=stage_id, executor_sha=executor_sha, now=time.time())
    observed_gpu = torch.cuda.get_device_name(0).upper() if torch.cuda.is_available() and torch.cuda.device_count() == 1 else ""
    if "H200" not in observed_gpu or not torch.cuda.is_bf16_supported():
        raise ContinuationError("exactly one bf16 H200 is required")
    usable_seconds = min((MAXIMUM_TOTAL_COMPUTE_CHARGE_USD - PERSISTENCE_RESERVE_USD) / TOTAL_RATE_USD_PER_SECOND, MAXIMUM_GPU_CHARGE_USD / GPU_RATE_USD_PER_SECOND, HARD_WALL_SECONDS)
    if usable_seconds < 600:
        raise ContinuationError("cost cap cannot cover the preregistered guarded continuation")
    deadline = started + usable_seconds
    _consume_authorization(authorization, run_id, stage_id)
    proven._remote_model_tree()

    processor = AutoProcessor.from_pretrained(proven.REMOTE_MODEL_DIR, local_files_only=True, trust_remote_code=False)
    tokenizer = processor.tokenizer
    if tokenizer.convert_tokens_to_ids(TARGET_TERMINATOR_TOKEN) != TARGET_TERMINATOR_ID or tokenizer.pad_token_id != 0:
        raise ContinuationError("remote tokenizer identity drifted")
    for raw, encoded in zip(bundle["rows"]["train_examples"], bundle["rows"]["train_encoded"], strict=True):
        replay = proven._reencode(tokenizer, raw)
        if any(encoded.get(field) != value for field, value in replay.items()):
            raise ContinuationError(f"exact chat-template replay mismatch: {raw['example_id']}")
    for role in PANEL_ROLES:
        for row in bundle["panels"][role]:
            proven._reencode(tokenizer, row)

    runs_root = Path("/continuation_output/runs").resolve()
    run_root = (runs_root / run_id).resolve()
    if run_root.parent != runs_root or not _valid_run_id(run_id):
        raise ContinuationError("run ID escapes continuation root")
    stage_source = authorization["stage_source"]
    if stage_id == "stage_a":
        source_adapter = _verify_source_step50()
        try:
            run_root.mkdir(parents=True, exist_ok=False)
        except FileExistsError as exc:
            raise ContinuationError("fresh Stage-A continuation run already exists") from exc
        start_step, end_step = 0, STAGE_A_END
        source_checkpoint = None
    else:
        if not run_root.is_dir():
            raise ContinuationError("Stage B requires the existing Stage-A run")
        eligibility_path = run_root / "stage-b-eligibility.json"
        eligibility = read_json(eligibility_path, str(stage_source["stage_b_eligibility_sha256"]))
        if not (
            valid_contract(eligibility, "eligibility_id")
            and eligibility["eligibility_id"] == stage_source["stage_b_eligibility_id"]
            and eligibility.get("status") == "stage_b_eligible_stable_or_improving_boundary"
            and eligibility.get("stage_a_trajectory_id") == stage_source["stage_a_trajectory_id"]
            and eligibility.get("stage_a_trajectory_sha256") == stage_source["stage_a_trajectory_sha256"]
            and eligibility.get("stage_b_authorization_allowed") is True
        ):
            raise ContinuationError("durable Stage-B eligibility receipt drifted")
        source_checkpoint = run_root / "checkpoints" / f"step-{STAGE_A_END:07d}"
        source_marker = _verify_checkpoint(source_checkpoint, bundle, run_id, executor_sha)
        if not (
            source_marker["marker_id"] == stage_source["checkpoint_marker_id"]
            and sha_file(source_checkpoint / "complete.json") == stage_source["checkpoint_marker_sha256"]
            and source_marker["checkpoint_tree_sha256"] == stage_source["checkpoint_tree_sha256"]
            and source_marker["adapter_tree_sha256"] == stage_source["adapter_tree_sha256"]
        ):
            raise ContinuationError("Stage-B checkpoint differs from its fresh authorization")
        source_adapter = source_checkpoint / "adapter"
        start_step, end_step = STAGE_A_END, STAGE_B_END

    random.seed(SEED)
    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cuda.matmul.allow_tf32 = True
    quantization = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.bfloat16)
    base = AutoModelForMultimodalLM.from_pretrained(proven.REMOTE_MODEL_DIR, local_files_only=True, trust_remote_code=False, quantization_config=quantization, device_map={"": 0}, dtype=torch.bfloat16, low_cpu_mem_usage=True)
    base.config.use_cache = False
    if hasattr(base.config, "text_config"):
        base.config.text_config.use_cache = False
    matched = [name for name, _ in base.named_modules() if re.fullmatch(proven.TARGET_MODULES_REGEX, name)]
    if len(matched) != proven.EXPECTED_TARGET_MODULES:
        raise ContinuationError("LoRA module topology drifted")
    base = prepare_model_for_kbit_training(base, use_gradient_checkpointing=True)
    model = PeftModel.from_pretrained(base, str(source_adapter), is_trainable=True)
    model.gradient_checkpointing_enable()
    trainable = [parameter for parameter in model.parameters() if parameter.requires_grad]
    if len(trainable) != proven.EXPECTED_LORA_TENSORS or sum(parameter.numel() for parameter in trainable) != proven.EXPECTED_TRAINABLE_PARAMETERS:
        raise ContinuationError("trainable LoRA topology drifted")

    # Stage A is deliberately fresh: no optimizer/scheduler/RNG file is read
    # from the immutable step-50 source. Stage B resumes only Stage A's state.
    optimizer = torch.optim.AdamW(trainable, lr=LEARNING_RATE / WARMUP_STEPS, betas=(0.9, 0.999), weight_decay=0.01)
    if stage_id == "stage_a":
        scheduler_state = {"kind": "manual_constant_with_linear_warmup", "base_learning_rate": LEARNING_RATE, "warmup_steps": WARMUP_STEPS, "last_completed_optimizer_step": 0, "initialized_fresh_from_source_adapter_only": True}
        ledgers: dict[str, Any] = {"loss": [], "gradient": [], "learning_rate": [], "runtime": [], "vram": [], "quality_gates": []}
        cursor = 0
        processed_tokens = 0
    else:
        assert source_checkpoint is not None
        state = torch.load(source_checkpoint / "fresh_optimizer_scheduler_rng.pt", map_location="cpu", weights_only=False)
        optimizer.load_state_dict(state["optimizer"])
        scheduler_state = dict(state["scheduler"])
        cursor = int(state["sampler_cursor"])
        processed_tokens = int(state["processed_native_tokens"])
        if not (cursor == STAGE_A_END and scheduler_state["last_completed_optimizer_step"] == STAGE_A_END and scheduler_state["base_learning_rate"] == LEARNING_RATE):
            raise ContinuationError("Stage-B optimizer/scheduler cursor drifted")
        random.setstate(state["python_rng"])
        torch.set_rng_state(state["torch_rng"])
        torch.cuda.set_rng_state_all(state["cuda_rng"])
        ledgers = read_json(source_checkpoint / "ledgers.json")
        trajectory = read_json(run_root / "trajectory.json", str(stage_source["stage_a_trajectory_sha256"]))
        if not (valid_contract(trajectory, "trajectory_id") and trajectory["trajectory_id"] == stage_source["stage_a_trajectory_id"] and trajectory.get("latest_optimizer_step") == STAGE_A_END):
            raise ContinuationError("Stage-B full trajectory receipt drifted")
        ledgers["quality_gates"] = list(trajectory["quality_gates"])
    recent_steps: list[float] = []
    encoded_by_id = {str(row["example_id"]): row for row in bundle["rows"]["train_encoded"]}
    span_by_id = {str(row["example_id"]): row for row in bundle["rows"]["loss_weight_spans"]}
    baseline: dict[str, Any] | None = None
    running_best: dict[str, Any] | None = None
    best_receipt: dict[str, Any] | None = None
    consecutive_soft_regressions = 0

    def checkpoint(kind: str) -> tuple[Path, dict[str, Any]]:
        return _publish_checkpoint(model=model, optimizer=optimizer, scheduler_state=scheduler_state, run_root=run_root, bundle=bundle, run_id=run_id, cursor=cursor, processed_tokens=processed_tokens, ledgers=ledgers, started=started, kind=kind)

    try:
        if stage_id == "stage_a":
            checkpoint_path, checkpoint_marker = checkpoint("immutable_step50_adapter_fresh_optimizer_step0")
            baseline = _publish_evaluation(model=model, tokenizer=tokenizer, bundle=bundle, run_root=run_root, run_id=run_id, step=0, checkpoint_path=checkpoint_path, checkpoint_marker=checkpoint_marker, deadline=deadline - PERSISTENCE_RESERVE_USD / TOTAL_RATE_USD_PER_SECOND, started=started)
            running_best = baseline
            best_receipt = _publish_best(run_root, 0, baseline, checkpoint_path, checkpoint_marker)
        else:
            baseline = read_json(run_root / "evaluations" / "step-0000000" / "summary.json")
            best_receipt = read_json(run_root / "best-checkpoint.json")
            if not (valid_contract(baseline, "evaluation_id") and valid_contract(best_receipt, "best_id")):
                raise ContinuationError("Stage-B baseline or best receipt drifted")
            best_step = int(best_receipt["optimizer_step"])
            running_best = read_json(run_root / "evaluations" / f"step-{best_step:07d}" / "summary.json")
            if running_best["evaluation_id"] != best_receipt["evaluation_id"]:
                raise ContinuationError("Stage-B running-best evaluation drifted")
        model.train()
        for position in range(start_step, end_step):
            step = bundle["rows"]["optimizer_steps"][position]
            step_number = position + 1
            _charge_guard(started, deadline, recent_steps, step_number in TRAINING_GATE_STEPS)
            ids = [str(value) for value in step["example_ids"]]
            step_rows = [encoded_by_id[value] for value in ids]
            spans = [(int(span_by_id[value]["verdict_value_span_start"]), int(span_by_id[value]["verdict_value_span_end"])) for value in ids]
            if len(step_rows) != EFFECTIVE_BATCH_SIZE or len(set(step["parent_ids"])) != EFFECTIVE_BATCH_SIZE:
                raise ContinuationError("runtime optimizer step repeats a parent")
            lr = LEARNING_RATE * min(step_number / WARMUP_STEPS, 1.0)
            for group in optimizer.param_groups:
                group["lr"] = lr
            step_started = time.perf_counter()
            torch.cuda.empty_cache()
            torch.cuda.reset_peak_memory_stats(0)
            before_memory = torch.cuda.memory_stats(0)
            optimizer.zero_grad(set_to_none=True)
            losses: list[float] = []
            for micro in range(GRADIENT_ACCUMULATION):
                start = micro * MICRO_BATCH_SIZE
                batch = proven._collate(step_rows[start:start + MICRO_BATCH_SIZE], tokenizer.pad_token_id)
                labels = batch.pop("labels")
                with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
                    loss = continuation_causal_loss(model(**batch).logits, labels, spans[start:start + MICRO_BATCH_SIZE])
                if not bool(torch.isfinite(loss).item()):
                    raise ContinuationError("nonfinite training loss")
                (loss / GRADIENT_ACCUMULATION).backward()
                losses.append(float(loss.detach().cpu()))
            gradient = float(torch.nn.utils.clip_grad_norm_(trainable, 1.0).detach().cpu())
            if not math.isfinite(gradient):
                raise ContinuationError("nonfinite gradient norm")
            optimizer.step()
            elapsed = time.perf_counter() - step_started
            after_memory = torch.cuda.memory_stats(0)
            peak_allocated = int(torch.cuda.max_memory_allocated(0))
            peak_reserved = int(torch.cuda.max_memory_reserved(0))
            retry_delta = int(after_memory.get("num_alloc_retries", 0)) - int(before_memory.get("num_alloc_retries", 0))
            oom_delta = int(after_memory.get("num_ooms", 0)) - int(before_memory.get("num_ooms", 0))
            if peak_allocated > MAXIMUM_PEAK_VRAM_BYTES or peak_reserved > MAXIMUM_PEAK_VRAM_BYTES or retry_delta != 0 or oom_delta != 0:
                raise ContinuationError("VRAM/allocator/OOM kill switch fired")
            cursor = step_number
            processed_tokens += int(step["native_tokens"])
            scheduler_state["last_completed_optimizer_step"] = cursor
            recent_steps.append(elapsed)
            ledgers["loss"].append({"optimizer_step": cursor, "mean_microbatch_loss": statistics.fmean(losses), "verdict_mass": VERDICT_LOSS_MASS, "finite": True})
            ledgers["gradient"].append({"optimizer_step": cursor, "preclip_l2_norm": gradient, "clip_max": 1.0, "finite": True})
            ledgers["learning_rate"].append({"optimizer_step": cursor, "learning_rate": lr})
            ledgers["runtime"].append({"optimizer_step": cursor, "runtime_seconds": elapsed, "native_tokens": step["native_tokens"], "native_tokens_per_second": int(step["native_tokens"]) / elapsed})
            ledgers["vram"].append({"optimizer_step": cursor, "peak_allocated_vram_bytes": peak_allocated, "peak_reserved_vram_bytes": peak_reserved, "allocation_retry_delta": retry_delta, "oom_delta": oom_delta})
            if cursor in TRAINING_GATE_STEPS:
                checkpoint_path, checkpoint_marker = checkpoint(f"pre_strict_gate_{cursor}")
                evaluation = _publish_evaluation(model=model, tokenizer=tokenizer, bundle=bundle, run_root=run_root, run_id=run_id, step=cursor, checkpoint_path=checkpoint_path, checkpoint_marker=checkpoint_marker, deadline=deadline - PERSISTENCE_RESERVE_USD / TOTAL_RATE_USD_PER_SECOND, started=started)
                assert baseline is not None and running_best is not None
                catastrophic = catastrophic_regression_gate(baseline, evaluation)
                soft = soft_regression_gate(baseline, running_best, evaluation)
                consecutive_soft_regressions = 0 if soft["all_gates_passed"] else consecutive_soft_regressions + 1
                gate = {
                    "status": "catastrophic_stop" if not catastrophic["all_gates_passed"] else ("soft_patience_exhausted" if consecutive_soft_regressions >= SOFT_REGRESSION_PATIENCE else ("soft_regression_watch" if not soft["all_gates_passed"] else "stable_or_improving")),
                    "catastrophic": catastrophic,
                    "soft": soft,
                    "consecutive_soft_regressions": consecutive_soft_regressions,
                    "soft_regression_patience": SOFT_REGRESSION_PATIENCE,
                    "all_gates_passed": catastrophic["all_gates_passed"] and consecutive_soft_regressions < SOFT_REGRESSION_PATIENCE,
                }
                ledgers["quality_gates"].append({"optimizer_step": cursor, **gate})
                atomic_json(run_root / "evaluations" / f"step-{cursor:07d}" / "guarded-trajectory-gate.json", content_contract(gate | {"version": VERSION, "optimizer_step": cursor}, "gate_id"))
                trajectory_receipt = content_contract({"version": VERSION, "status": "full_guarded_continuation_trajectory", "run_id": run_id, "latest_optimizer_step": cursor, "quality_gates": ledgers["quality_gates"], "promotion_allowed": False}, "trajectory_id")
                atomic_json(run_root / "trajectory.json", trajectory_receipt)
                output_volume.commit()
                if not catastrophic["all_gates_passed"] or consecutive_soft_regressions >= SOFT_REGRESSION_PATIENCE:
                    raise QualityRegressionStop(cursor, gate)
                if soft["all_gates_passed"] and best_rank(evaluation) > best_rank(running_best):
                    running_best = evaluation
                    best_receipt = _publish_best(run_root, cursor, evaluation, checkpoint_path, checkpoint_marker)
                if cursor == STAGE_A_END and soft["all_gates_passed"]:
                    eligibility = content_contract(
                        {
                            "version": VERSION,
                            "status": "stage_b_eligible_stable_or_improving_boundary",
                            "run_id": run_id,
                            "stage_a_optimizer_step": STAGE_A_END,
                            "stage_a_checkpoint_marker_id": checkpoint_marker["marker_id"],
                            "stage_a_checkpoint_marker_sha256": sha_file(checkpoint_path / "complete.json"),
                            "stage_a_checkpoint_tree_sha256": checkpoint_marker["checkpoint_tree_sha256"],
                            "stage_a_adapter_tree_sha256": checkpoint_marker["adapter_tree_sha256"],
                            "stage_a_evaluation_id": evaluation["evaluation_id"],
                            "stage_a_gate_status": gate["status"],
                            "stage_a_trajectory_id": trajectory_receipt["trajectory_id"],
                            "stage_a_trajectory_sha256": sha_file(run_root / "trajectory.json"),
                            "consecutive_soft_regressions": consecutive_soft_regressions,
                            "stage_b_authorization_allowed": True,
                            "fresh_cost_authorization_required": True,
                            "promotion_allowed": False,
                        },
                        "eligibility_id",
                    )
                    atomic_json(run_root / "stage-b-eligibility.json", eligibility)
                    output_volume.commit()
                model.train()
        if best_receipt is None:
            raise ContinuationError("best checkpoint receipt missing")
        total, gpu, seconds = _bounded_cost(started)
        stage_b_eligible = (run_root / "stage-b-eligibility.json").is_file() if stage_id == "stage_a" else False
        return _write_terminal(run_root, {"status": f"completed_{stage_id}_guarded_continuation_best_not_last_retained", "stage_id": stage_id, "run_id": run_id, "optimizer_step": cursor, "processed_native_tokens": processed_tokens, "best_checkpoint": best_receipt, "last_checkpoint_is_implicitly_best": False, "stage_b_eligible": stage_b_eligible, "stage_b_requires_fresh_cost_authorization": stage_id == "stage_a" and stage_b_eligible, "bounded_total_compute_cost_usd": total, "bounded_gpu_cost_usd": gpu, "bounded_seconds_including_persistence": seconds, "post_training_confirmatory600_local_gate_required": True, "quality_claim_allowed": False}, stage_id)
    except QualityRegressionStop as stop:
        total, gpu, seconds = _bounded_cost(started)
        return _write_terminal(run_root, {"status": "stopped_by_guarded_quality_policy", "stage_id": stage_id, "run_id": run_id, "optimizer_step": stop.step, "gate": stop.gate, "best_checkpoint": best_receipt, "rejected_last_checkpoint_preserved": True, "bounded_total_compute_cost_usd": total, "bounded_gpu_cost_usd": gpu, "bounded_seconds_including_persistence": seconds, "quality_claim_allowed": False}, stage_id)
    except CleanBudgetStop as stop:
        checkpoint_path, checkpoint_marker = checkpoint("clean_budget_stop")
        total, gpu, seconds = _bounded_cost(started)
        return _write_terminal(run_root, {"status": "clean_budget_stop_no_automatic_resume", "stage_id": stage_id, "reason": str(stop), "run_id": run_id, "optimizer_step": cursor, "checkpoint_marker_id": checkpoint_marker["marker_id"], "checkpoint_marker_sha256": sha_file(checkpoint_path / "complete.json"), "best_checkpoint": best_receipt, "bounded_total_compute_cost_usd": total, "bounded_gpu_cost_usd": gpu, "bounded_seconds_including_persistence": seconds, "resume_requires_separately_preregistered_fresh_authorization": True, "quality_claim_allowed": False}, stage_id)


def _guarded_run(run_id: str, stage_id: str, authorization: Mapping[str, Any]) -> dict[str, Any]:
    try:
        return _run_remote(run_id, stage_id, authorization)
    except BaseException as exc:
        try:
            failure = content_contract({"version": VERSION, "status": "fatal_continuation_failure", "run_id": run_id, "stage_id": stage_id, "exception_type": type(exc).__name__, "exception_message": str(exc), "traceback": traceback.format_exc(), "authorization_consumed": True, "promotion_allowed": False, "failed_epoch": time.time()}, "failure_id")
            atomic_json(Path("/continuation_output/failures") / f"{run_id}__{stage_id}__{failure['failure_id']}.json", failure)
            output_volume.commit()
        finally:
            raise


app = modal.App("bichar-gemma4-e4b-step50-guarded-continuation-v1")
model_volume = modal.Volume.from_name(MODEL_VOLUME, create_if_missing=False)
source_volume = modal.Volume.from_name(SOURCE_CHECKPOINT_VOLUME, create_if_missing=False)
output_volume = modal.Volume.from_name(OUTPUT_CHECKPOINT_VOLUME, create_if_missing=True)
continuation_image = _select_image()


@app.function(
    image=continuation_image,
    gpu="H200",
    cpu=CPU_CORES,
    memory=65_536,
    timeout=HARD_WALL_SECONDS,
    retries=0,
    block_network=True,
    restrict_modal_access=True,
    single_use_containers=True,
    max_containers=1,
    volumes={"/models": model_volume.read_only(), "/step50_source": source_volume.read_only(), "/continuation_output": output_volume},
)
def train_step50_guarded_continuation_h200(run_id: str, stage_id: str, authorization: dict[str, Any]) -> dict[str, Any]:
    return _guarded_run(run_id, stage_id, authorization)


@app.local_entrypoint()
def launch_one(run_id: str, stage_id: str, authorization_path: str, detached_attestation: bool = False) -> None:
    if not detached_attestation or os.environ.get("BICHAR_MODAL_DETACHED") != "1":
        raise ContinuationError("detached launch attestation is missing")
    plan = build_local_plan(DEFAULT_ACCEPTANCE, Path(authorization_path))
    if not plan["launch_ready"] or plan["run_id"] != run_id or plan["stage_id"] != stage_id:
        raise ContinuationError(f"continuation plan is NO-GO: {plan['blockers']}")
    authorization = read_json(Path(authorization_path))
    call = train_step50_guarded_continuation_h200.spawn(run_id=run_id, stage_id=stage_id, authorization=authorization)
    print(canonical({"version": VERSION, "status": "exactly_one_detached_continuation_stage_call_spawned", "run_id": run_id, "stage_id": stage_id, "call_id": call.object_id, "authorization_id": authorization["authorization_id"], "second_call_spawned": False, "local_tether_waited": False}))


def _parser() -> argparse.ArgumentParser:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--plan", action="store_true")
    parser.add_argument("--acceptance", type=Path)
    parser.add_argument("--authorization", type=Path)
    parser.add_argument("--write-static-acceptance", action="store_true")
    parser.add_argument("--acceptance-output", type=Path, default=DEFAULT_ACCEPTANCE)
    parser.add_argument("--emit-safe-commands", action="store_true")
    parser.add_argument("--commands-output", type=Path, default=DEFAULT_COMMANDS)
    parser.add_argument("--mint-authorization", action="store_true")
    parser.add_argument("--workspace-receipt", type=Path)
    parser.add_argument("--run-id")
    parser.add_argument("--stage-id", choices=STAGE_IDS)
    parser.add_argument("--stage-a-checkpoint", type=Path)
    parser.add_argument("--stage-a-eligibility", type=Path)
    parser.add_argument("--output-authorization", type=Path)
    return parser


def main(argv: Sequence[str] | None = None) -> int:
    args = _parser().parse_args(argv)
    if args.write_static_acceptance:
        value = write_static_acceptance(args.acceptance_output)
    elif args.emit_safe_commands:
        if args.acceptance is None:
            raise ContinuationError("--acceptance is required")
        value = emit_safe_commands(args.acceptance, args.commands_output)
    elif args.mint_authorization:
        if args.acceptance is None or args.workspace_receipt is None or args.run_id is None or args.stage_id is None or args.output_authorization is None:
            raise ContinuationError("minting requires acceptance, workspace receipt, run ID, stage ID, and output")
        value = mint_authorization(acceptance_path=args.acceptance, workspace_receipt_path=args.workspace_receipt, run_id=args.run_id, stage_id=args.stage_id, stage_a_checkpoint_path=args.stage_a_checkpoint, stage_a_eligibility_path=args.stage_a_eligibility)
        atomic_json(args.output_authorization, value)
    else:
        value = build_local_plan(args.acceptance, args.authorization)
    print(json.dumps(value, ensure_ascii=False, sort_keys=True, indent=2))
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


## Multi-arm sweep campaign

The following code cell is the exact fail-closed campaign executor. It starts
every arm from the same immutable adapter with a fresh optimizer, evaluates
all four arms at steps 0/10/20, and retains only earned continuations. Paid
execution requires a one-use authorization that is deliberately absent.

In [ ]:
%%writefile training_code/modal_train_gemma4_e4b_step50_sweep_campaign_v1.py
"""One-call, fail-closed 2x2 continuation sweep from immutable Gemma step 50.

This executor is deliberately launch-disabled by default.  It consumes a sealed
7 canonical + 1 owned-synthetic mixed pack and evaluates four arms at steps
0/10/20, retains the best two through step 60, and retains the winner through
step 100.  Every arm starts from the same immutable adapter with a fresh AdamW
optimizer.  A warmup/stable/decay (WSD) learning-rate schedule is used.

The static commands never request a GPU.  The sole paid entrypoint requires a
fresh, hash-bound, one-use authorization and an explicit detached attestation.
"""

from __future__ import annotations

import argparse
import ast
from collections import Counter
from collections.abc import Mapping, Sequence
import hashlib
import gc
import json
import math
import os
from pathlib import Path
import random
import re
import shutil
import statistics
import time
import traceback
from typing import Any
import uuid

import modal

from pipeline import modal_train_gemma4_e4b_step50_guarded_continuation_v1 as guarded


ROOT = Path(__file__).resolve().parents[1]
VERSION = "bichar-gemma4-e4b-step50-sweep-campaign-v1"
AUTH_VERSION = VERSION + "-authorization"
ACCEPTANCE_VERSION = VERSION + "-static-acceptance"
PACK = ROOT / "artifacts/training/gemma4_e4b_step50_continuation_sweep_v1_20260720"
MANIFEST = PACK / "manifest.json"
SPEC = PACK / "sweep_spec.json"
MANIFEST_SHA256 = "d490d354498dac2d596fe08233ddcf8186eafed4b35797f01538ce3f2dc23110"
MANIFEST_ID = "d86c537528561c17c3cb0e8e1ba35476e4dc9885d32cafa6ae658a27ca974dcb"
SPEC_SHA256 = "91da06cd4195b6dbfaf43910bdbd5694b36450e5c9140c640671548cab5afc86"
SPEC_ID = "5593f2dc3df3bc8f2c25ce49531bc8eebcf7d43c7d704409c21ae58b1538579c"
DEFAULT_ACCEPTANCE = ROOT / "artifacts/status/GEMMA4_E4B_STEP50_SWEEP_CAMPAIGN_ACCEPTANCE_20260720.json"
DEFAULT_COMMANDS = ROOT / "artifacts/status/GEMMA4_E4B_STEP50_SWEEP_CAMPAIGN_SAFE_COMMANDS_20260720.json"

PACK_FILES = {
    "train_examples": "train_examples.jsonl",
    "train_encoded": "train_encoded.jsonl",
    "optimizer_steps": "optimizer_steps.jsonl",
    "loss_weight_spans": "loss_weight_spans.jsonl",
}
EXPECTED_ARTIFACT_BINDINGS = {
    "loss_weight_spans.jsonl": {"sha256": "aecadcfd2d8368bb4fced3620ed52ca6eaed61b4c341bc51e96b68f56d1e5888", "rows": 800},
    "optimizer_steps.jsonl": {"sha256": "ff52626dc35fec380b4f8dd2a902d46466307b3bd2dd859df4ec10a42886690f", "rows": 100},
    "train_encoded.jsonl": {"sha256": "7b3f6efa8de331d6f90b2cc4a4f7ecf9dcdcd4d1d737fb9ec3508e70261cbad8", "rows": 800},
    "train_examples.jsonl": {"sha256": "a9112a1cdac1a2a53c29860015e631277edcdb2bdf81b7f1f3a919383f84b81b", "rows": 800},
}
EXPECTED_PANEL_BINDINGS = {
    "calibration_examples": {"sha256": "d50fc84ad30d797976784966f838e33a16eec7311b19f486d7e691c66bc2a610", "rows": 32},
    "group_held_examples": {"sha256": "95112ad678a2b7689418c46a3b786a024560e17c9b1c51ae1fbb6ba938c56a3f", "rows": 32},
    "long_context_diagnostic_examples": {"sha256": "22876f37ae983fa5a71bbbc35c00d0c166b73c45e68ae9ebb078d8e0740dfa1b", "rows": 36},
    "source_held_examples": {"sha256": "aafcf41e9c90f6ee3cbac9b09c1778d22087b952b8142de6e4c792a966ffd084", "rows": 30},
}
PANEL_ROLES = guarded.PANEL_ROLES
VERDICTS = guarded.VERDICTS
ARMS = {
    "lr1e6_verdict25": {"learning_rate": 1e-6, "verdict_loss_mass": 0.25},
    "lr1e6_verdict45": {"learning_rate": 1e-6, "verdict_loss_mass": 0.45},
    "lr2e6_verdict25": {"learning_rate": 2e-6, "verdict_loss_mass": 0.25},
    "lr2e6_verdict45": {"learning_rate": 2e-6, "verdict_loss_mass": 0.45},
}
ALL_ARM_GATES = (0, 10, 20)
TOP2_GATES = (30, 40, 50, 60)
WINNER_GATES = (70, 80, 90, 100)
ALL_GATES = ALL_ARM_GATES + TOP2_GATES + WINNER_GATES
TOTAL_STEPS = 100
WARMUP_END = 4
STABLE_END = 60
FINAL_LR_RATIO = 0.1
EFFECTIVE_BATCH_SIZE = 8
CANONICAL_PER_STEP = 7
SYNTHETIC_PER_STEP = 1
SOFT_PATIENCE = 2
MATERIAL_MACRO_REGRESSION = 0.03
SEED = 20260720
HARD_WALL_SECONDS = 1800
MAXIMUM_TOTAL_COMPUTE_CHARGE_USD = 2.75
MINIMUM_OUTSIDE_RESERVE_USD = 10.0
OUTPUT_VOLUME = "bichar-gemma4-e4b-step50-sweep-campaign-v1"
RUN_ID_RE = re.compile(r"(?=.{1,96}$)[A-Za-z0-9](?:[A-Za-z0-9_.-]*[A-Za-z0-9])?")

canonical = guarded.canonical
sha_file = guarded.sha_file
read_json = guarded.read_json
read_jsonl = guarded.read_jsonl
atomic_json = guarded.atomic_json
content_contract = guarded.content_contract
valid_contract = guarded.valid_contract


class SweepError(ValueError):
    pass


def wsd_learning_rate(step: int, base_lr: float) -> float:
    """Warm up 1-4, remain stable 5-60, cosine decay 61-100 to 0.1x."""
    if step < 1 or step > TOTAL_STEPS or base_lr not in {1e-6, 2e-6}:
        raise SweepError("invalid WSD input")
    if step <= WARMUP_END:
        return base_lr * step / WARMUP_END
    if step <= STABLE_END:
        return base_lr
    progress = (step - STABLE_END) / (TOTAL_STEPS - STABLE_END)
    multiplier = FINAL_LR_RATIO + (1.0 - FINAL_LR_RATIO) * 0.5 * (1.0 + math.cos(math.pi * progress))
    return base_lr * multiplier


def _pack_identity() -> dict[str, str]:
    if not MANIFEST.is_file():
        raise SweepError(f"sealed mixed pack missing: {MANIFEST}")
    manifest = read_json(MANIFEST, MANIFEST_SHA256)
    sweep_spec = read_json(SPEC, SPEC_SHA256)
    if not (valid_contract(manifest, "manifest_id") and manifest["manifest_id"] == MANIFEST_ID and valid_contract(sweep_spec, "spec_id") and sweep_spec["spec_id"] == SPEC_ID and manifest["artifacts"]["sweep_spec.json"]["sha256"] == SPEC_SHA256):
        raise SweepError("mixed manifest content contract is invalid")
    return {"manifest_id": MANIFEST_ID, "manifest_sha256": MANIFEST_SHA256, "spec_id": SPEC_ID, "spec_sha256": SPEC_SHA256}


def _artifact_path(spec: Mapping[str, Any]) -> Path:
    path = (ROOT / str(spec["path"])).resolve()
    if ROOT.resolve() not in path.parents:
        raise SweepError("artifact escapes project root")
    return path


def _row_mix_kind(row: Mapping[str, Any]) -> str:
    values = [
        row.get("sweep_origin"), row.get("mix_role"), row.get("training_mix_role"), row.get("origin_role"),
        row.get("source_kind"), row.get("provenance", {}).get("source_kind") if isinstance(row.get("provenance"), Mapping) else None,
    ]
    text = " ".join(str(value).casefold() for value in values if value is not None)
    return "synthetic" if "synthetic" in text else "canonical"


def load_mixed_pack(*, verify: bool = True) -> dict[str, Any]:
    identity = _pack_identity()
    manifest = read_json(MANIFEST, identity["manifest_sha256"])
    sweep_spec = read_json(SPEC, SPEC_SHA256)
    if manifest.get("private_competition_labels_included") is not False:
        raise SweepError("private labels are forbidden")
    rows: dict[str, list[dict[str, Any]]] = {}
    for role, filename in PACK_FILES.items():
        artifact_spec = manifest.get("artifacts", {}).get(filename)
        if not isinstance(artifact_spec, Mapping):
            raise SweepError(f"missing sealed artifact: {filename}")
        path = _artifact_path(artifact_spec)
        if verify and sha_file(path) != artifact_spec.get("sha256"):
            raise SweepError(f"artifact hash drifted: {filename}")
        rows[role] = read_jsonl(path)
        if len(rows[role]) != int(artifact_spec.get("rows", -1)):
            raise SweepError(f"artifact row count drifted: {filename}")
    examples = rows["train_examples"]
    by_id = {str(row["example_id"]): row for row in examples}
    ids = [str(row["example_id"]) for row in examples]
    if len(ids) != 800 or len(by_id) != 800:
        raise SweepError("mixed pack must contain 800 unique examples")
    if ids != [str(row["example_id"]) for row in rows["train_encoded"]] or ids != [str(row["example_id"]) for row in rows["loss_weight_spans"]]:
        raise SweepError("parallel training ledgers are not aligned")
    steps = rows["optimizer_steps"]
    if len(steps) != TOTAL_STEPS:
        raise SweepError("mixed pack must contain exactly 100 optimizer steps")
    flattened: list[str] = []
    for expected_step, step in enumerate(steps, 1):
        step_ids = [str(value) for value in step.get("example_ids", [])]
        if len(step_ids) != EFFECTIVE_BATCH_SIZE or len(set(step_ids)) != EFFECTIVE_BATCH_SIZE:
            raise SweepError(f"step {expected_step} is not an eight-parent batch")
        kinds = Counter(_row_mix_kind(by_id[value]) for value in step_ids)
        if kinds != Counter({"canonical": CANONICAL_PER_STEP, "synthetic": SYNTHETIC_PER_STEP}):
            raise SweepError(f"step {expected_step} violates exact 7+1 mix")
        flattened.extend(step_ids)
    if flattened != ids:
        raise SweepError("optimizer schedule does not exactly consume sealed order")
    panels: dict[str, list[dict[str, Any]]] = {}
    for role in PANEL_ROLES:
        panel_spec = manifest.get("panel_bindings", {}).get(role)
        if not isinstance(panel_spec, Mapping):
            raise SweepError(f"missing locked panel: {role}")
        path = _artifact_path(panel_spec)
        if verify and sha_file(path) != panel_spec.get("sha256"):
            raise SweepError(f"locked panel drifted: {role}")
        panels[role] = read_jsonl(path)
        if len(panels[role]) != int(panel_spec.get("rows", -1)):
            raise SweepError(f"locked panel count drifted: {role}")
    train_keys = {str(row.get(key)) for row in examples for key in ("example_id", "parent_id", "record_id") if row.get(key) is not None}
    panel_keys = {str(value) for panel in panels.values() for row in panel for value in (row.get("example_id"), row.get("parent_id"), row.get("question_group_id")) if value is not None}
    if train_keys & panel_keys:
        raise SweepError("train/evaluation identity overlap")
    expected_arms = {arm: {"base_learning_rate": values["learning_rate"], "verdict_loss_mass": values["verdict_loss_mass"]} for arm, values in ARMS.items()}
    execution = sweep_spec["execution"]
    if not (sweep_spec["arms"] == expected_arms and sweep_spec["curriculum"]["canonical_gold_per_batch"] == 7 and sweep_spec["curriculum"]["owned_synthetic_per_batch"] == 1 and sweep_spec["successive_halving"]["material_group_source_mean_drop"] == MATERIAL_MACRO_REGRESSION and sweep_spec["successive_halving"]["aggregate_soft_patience"] == SOFT_PATIENCE and execution["hard_aggregate_compute_cap_usd"] == MAXIMUM_TOTAL_COMPUTE_CHARGE_USD and execution["mode"] == "fresh_hash_bound_base_reload_for_each_arm_and_resume_phase_inside_same_single_use_container" and execution["base_reuse_enabled"] is False and execution["maximum_base_loads"] == 7 and execution["exact_offline_model_tree_verification_required_before_each_base_load"] is True and execution["delete_model_optimizer_trainables_and_empty_cuda_cache_after_each_phase"] is True):
        raise SweepError("sealed sweep specification drifted")
    for arm, values in ARMS.items():
        table = sweep_spec["scheduler"]["tables"][arm]
        if len(table) != TOTAL_STEPS or any(entry["optimizer_step"] != step or abs(entry["learning_rate"] - wsd_learning_rate(step, values["learning_rate"])) > 1e-18 for step, entry in enumerate(table, 1)):
            raise SweepError(f"sealed WSD table drifted: {arm}")
    return {"identity": identity, "manifest": manifest, "spec": sweep_spec, "rows": rows, "panels": panels}


def campaign_rounds() -> list[dict[str, Any]]:
    return [
        {"round": "screen_all_four", "arms": sorted(ARMS), "train_to": 20, "evaluate": list(ALL_ARM_GATES), "retain": 2},
        {"round": "compare_top_two", "arms": "top2_from_step20", "train_to": 60, "evaluate": list(TOP2_GATES), "retain": 1},
        {"round": "extend_winner", "arms": "winner_from_step60", "train_to": 100, "evaluate": list(WINNER_GATES), "retain": 1},
    ]


def aggregate_rank(evaluation: Mapping[str, Any]) -> tuple[float, ...]:
    """Rank group/source first, with long-context and calibration tie-breakers."""
    panels = evaluation["calibrated_panels"]
    group = panels["group_held_examples"]
    source = panels["source_held_examples"]
    long_panel = panels["long_context_diagnostic_examples"]
    calibration = panels["calibration_examples"]
    return (
        (float(group["macro_f1"]) + float(source["macro_f1"])) / 2,
        min(float(group["macro_f1"]), float(source["macro_f1"])),
        float(long_panel["macro_f1"]),
        -float(calibration["expected_calibration_error"]),
        -float(calibration["brier_score"]),
        -float(evaluation["optimizer_step"]),
    )


def material_aggregate_regression(best: Mapping[str, Any], candidate: Mapping[str, Any]) -> bool:
    if int(candidate["optimizer_step"]) <= 20:
        return False
    before, after = aggregate_rank(best), aggregate_rank(candidate)
    return after[0] < before[0] - MATERIAL_MACRO_REGRESSION or after[1] < before[1] - MATERIAL_MACRO_REGRESSION


def eligible_best_update(best: Mapping[str, Any], candidate: Mapping[str, Any], catastrophic: Mapping[str, Any], stopped: bool) -> bool:
    """A rejected or stopped checkpoint can never become best."""
    return catastrophic.get("all_gates_passed") is True and not stopped and aggregate_rank(candidate) > aggregate_rank(best)


def static_acceptance() -> dict[str, Any]:
    bundle = load_mixed_pack(verify=True)
    source = Path(__file__).read_text(encoding="utf-8")
    checks = {
        "sealed_hash_bound_7plus1_pack": len(bundle["rows"]["train_examples"]) == 800,
        "embedded_artifact_bindings_equal_manifest": EXPECTED_ARTIFACT_BINDINGS == {name: {"sha256": spec["sha256"], "rows": spec.get("rows")} for name, spec in bundle["manifest"]["artifacts"].items() if name in PACK_FILES.values()} and EXPECTED_PANEL_BINDINGS == {role: {"sha256": spec["sha256"], "rows": spec["rows"]} for role, spec in bundle["manifest"]["panel_bindings"].items() if role in PANEL_ROLES},
        "exact_2x2_arms": {(v["learning_rate"], v["verdict_loss_mass"]) for v in ARMS.values()} == {(1e-6, .25), (1e-6, .45), (2e-6, .25), (2e-6, .45)},
        "immutable_step50_fresh_optimizer_each_arm": "fresh_optimizer_each_arm" in source,
        "wsd_warmup_stable_cosine_decay": all(abs(wsd_learning_rate(step, 2e-6) - expected) < 1e-15 for step, expected in ((1, .5e-6), (4, 2e-6), (5, 2e-6), (60, 2e-6), (100, .2e-6))),
        "all_to20_top2_to60_winner_to100": campaign_rounds()[0]["retain"] == 2 and campaign_rounds()[2]["train_to"] == 100,
        "exact_locked_panels": set(bundle["panels"]) == set(PANEL_ROLES),
        "catastrophic_immediate_material_patience_after20": "catastrophic_regression_gate" in source and SOFT_PATIENCE == 2 and "material_aggregate_regression" in source,
        "best_not_last": "best-checkpoint.json" in source,
        "one_container_network_blocked_retry0": all(token in source for token in ("block_network=True", "max_containers=1", "retries=0")),
        "aggregate_cost_cap_and_reserve": MAXIMUM_TOTAL_COMPUTE_CHARGE_USD == 2.75 and MINIMUM_OUTSIDE_RESERVE_USD == 10.0,
        "one_detached_h200_campaign": 'gpu="H200"' in source and sum(1 for node in ast.walk(ast.parse(source)) if isinstance(node, ast.Call) and isinstance(node.func, ast.Attribute) and node.func.attr == "spawn") == 1,
        "real_included_tensor_loop_no_absent_hook": dry_remote_simulation()["all_checks_passed"],
        "no_private_labels": bundle["manifest"].get("private_competition_labels_included") is False,
        "no_launch_authority": True,
    }
    if not all(checks.values()):
        raise SweepError(f"static acceptance failed: {[k for k, v in checks.items() if not v]}")
    return content_contract({
        "version": ACCEPTANCE_VERSION,
        "status": "ALL_STATIC_ACCEPTANCE_CHECKS_PASS_BUILD_ONLY_NO_LAUNCH_AUTHORITY",
        "executor_sha256": sha_file(Path(__file__)),
        **bundle["identity"],
        "checks": checks,
        "campaign_rounds": campaign_rounds(),
        "authorization_present": False,
        "gpu_requested": False,
        "paid_action_performed": False,
        "launch_performed": False,
    }, "acceptance_id")


def dry_remote_simulation() -> dict[str, Any]:
    """AST/data simulation of the paid call path; never loads weights or requests GPU."""
    load_mixed_pack(verify=True)
    tree = ast.parse(Path(__file__).read_text(encoding="utf-8"))
    functions = {node.name: node for node in tree.body if isinstance(node, ast.FunctionDef)}
    remote = functions.get("_remote_campaign")
    if remote is None:
        raise SweepError("remote campaign function missing")
    called_names = [node.func.id for node in ast.walk(remote) if isinstance(node, ast.Call) and isinstance(node.func, ast.Name)]
    source = Path(__file__).read_text(encoding="utf-8")
    remote_attrs = {node.func.attr for node in ast.walk(remote) if isinstance(node, ast.Call) and isinstance(node.func, ast.Attribute)}
    checks = {
        "real_tensor_loop_present": all(name in functions for name in ("_run_arm_phase", "_load_arm", "sweep_causal_loss")),
        "remote_invokes_arm_phase": called_names.count("_run_arm_phase") >= 3,
        "no_missing_delegated_hook": "run_sealed_" + "sweep_campaign" not in remote_attrs,
        "authorization_validated": "_validate_authorization" in called_names,
        "authorization_consumed_before_offline_model_tree": source.index("consumption.mkdir") < source.index("guarded.proven._remote_model_tree()"),
        "atomic_checkpoint_and_evaluation_present": all(name in functions for name in ("_atomic_checkpoint", "_atomic_evaluation")),
        "hash_bound_source_verification": "guarded._verify_source_step50()" in source,
        "isolated_base_reload_branch": "AutoModelForMultimodalLM.from_pretrained" in source and "finally:" in source and "torch.cuda.empty_cache()" in source,
    }
    if not all(checks.values()):
        raise SweepError(f"dry remote simulation failed: {[k for k, v in checks.items() if not v]}")
    return {"version": VERSION, "status": "DRY_REMOTE_CALL_PATH_VALID_NO_GPU_NO_MODEL_LOAD", "checks": checks, "all_checks_passed": True, "gpu_requested": False, "paid_action_performed": False}


def _validate_workspace(value: Mapping[str, Any], now: float) -> None:
    if not (valid_contract(value, "receipt_id") and value.get("owned_workspace_authorized") is True and value.get("no_active_containers_observed") is True and value.get("no_running_apps_observed") is True and float(value.get("available_credit_usd", 0)) >= MAXIMUM_TOTAL_COMPUTE_CHARGE_USD + MINIMUM_OUTSIDE_RESERVE_USD and now - float(value.get("captured_epoch", 0)) <= 900):
        raise SweepError("fresh workspace/cost receipt invalid")


def mint_authorization(acceptance_path: Path, workspace_path: Path, run_id: str) -> dict[str, Any]:
    bundle = load_mixed_pack(verify=True)
    acceptance = read_json(acceptance_path)
    workspace = read_json(workspace_path)
    now = time.time()
    _validate_workspace(workspace, now)
    if not valid_contract(acceptance, "acceptance_id") or acceptance.get("executor_sha256") != sha_file(Path(__file__)) or acceptance.get("manifest_sha256") != bundle["identity"]["manifest_sha256"]:
        raise SweepError("static acceptance is stale or invalid")
    if RUN_ID_RE.fullmatch(run_id) is None:
        raise SweepError("invalid run ID")
    artifacts = {name: {"sha256": spec["sha256"], "rows": spec.get("rows")} for name, spec in bundle["manifest"]["artifacts"].items() if name in PACK_FILES.values()}
    panels = {role: {"sha256": bundle["manifest"]["panel_bindings"][role]["sha256"], "rows": bundle["manifest"]["panel_bindings"][role]["rows"]} for role in PANEL_ROLES}
    return content_contract({
        "version": AUTH_VERSION,
        "status": "fresh_one_use_hash_bound_sweep_campaign_authorization",
        "executor_sha256": sha_file(Path(__file__)),
        **bundle["identity"],
        "artifact_bindings": artifacts,
        "panel_bindings": panels,
        "source_marker_id": guarded.SOURCE_MARKER_ID,
        "source_marker_sha256": guarded.SOURCE_MARKER_SHA256,
        "source_checkpoint_tree_sha256": guarded.SOURCE_CHECKPOINT_TREE_SHA256,
        "source_adapter_tree_sha256": guarded.SOURCE_ADAPTER_TREE_SHA256,
        "workspace_receipt": workspace,
        "run_id": run_id,
        "arms": ARMS,
        "campaign_rounds": campaign_rounds(),
        "issued_epoch": now,
        "expires_epoch": now + 900,
        "authorize_exactly_one_detached_h200_campaign": True,
        "fresh_optimizer_each_arm": True,
        "network_blocked_required": True,
        "retries": 0,
        "maximum_containers": 1,
        "maximum_total_compute_charge_usd": MAXIMUM_TOTAL_COMPUTE_CHARGE_USD,
        "minimum_outside_credit_reserve_usd": MINIMUM_OUTSIDE_RESERVE_USD,
        "private_competition_labels_allowed": False,
        "promotion_allowed": False,
    }, "authorization_id")


def _validate_authorization(value: Mapping[str, Any], run_id: str, now: float, executor_sha: str) -> dict[str, Any]:
    workspace = value.get("workspace_receipt", {})
    _validate_workspace(workspace, now)
    expected_artifacts = EXPECTED_ARTIFACT_BINDINGS
    expected_panels = EXPECTED_PANEL_BINDINGS
    if not (valid_contract(value, "authorization_id") and value.get("version") == AUTH_VERSION and value.get("executor_sha256") == executor_sha and value.get("manifest_id") == MANIFEST_ID and value.get("manifest_sha256") == MANIFEST_SHA256 and value.get("spec_id") == SPEC_ID and value.get("spec_sha256") == SPEC_SHA256 and value.get("artifact_bindings") == expected_artifacts and value.get("panel_bindings") == expected_panels and value.get("source_marker_id") == guarded.SOURCE_MARKER_ID and value.get("source_marker_sha256") == guarded.SOURCE_MARKER_SHA256 and value.get("source_checkpoint_tree_sha256") == guarded.SOURCE_CHECKPOINT_TREE_SHA256 and value.get("source_adapter_tree_sha256") == guarded.SOURCE_ADAPTER_TREE_SHA256 and value.get("run_id") == run_id and value.get("arms") == ARMS and value.get("campaign_rounds") == campaign_rounds() and float(value.get("issued_epoch", math.inf)) <= now <= float(value.get("expires_epoch", -math.inf)) and value.get("authorize_exactly_one_detached_h200_campaign") is True and value.get("fresh_optimizer_each_arm") is True and value.get("network_blocked_required") is True and value.get("retries") == 0 and value.get("maximum_containers") == 1 and value.get("maximum_total_compute_charge_usd") == MAXIMUM_TOTAL_COMPUTE_CHARGE_USD and value.get("minimum_outside_credit_reserve_usd") == MINIMUM_OUTSIDE_RESERVE_USD and value.get("private_competition_labels_allowed") is False):
        raise SweepError("campaign authorization is invalid")
    return dict(value)


def build_local_plan(acceptance_path: Path | None = None, authorization_path: Path | None = None) -> dict[str, Any]:
    blockers: list[str] = []
    try:
        bundle = load_mixed_pack(verify=True)
    except Exception as exc:
        bundle = None
        blockers.append(f"sealed_mixed_pack_invalid:{type(exc).__name__}")
    if acceptance_path is None:
        blockers.append("static_acceptance_missing")
    else:
        try:
            acceptance = read_json(acceptance_path)
            if not valid_contract(acceptance, "acceptance_id") or acceptance.get("executor_sha256") != sha_file(Path(__file__)):
                raise SweepError("stale acceptance")
        except Exception as exc:
            blockers.append(f"static_acceptance_invalid:{type(exc).__name__}")
    auth = None
    if authorization_path is None:
        blockers.append("fresh_one_use_authorization_missing")
    else:
        try:
            auth = read_json(authorization_path)
            _validate_authorization(auth, str(auth.get("run_id")), time.time(), sha_file(Path(__file__)))
            if bundle is None or auth.get("manifest_sha256") != bundle["identity"]["manifest_sha256"]:
                raise SweepError("authorization/pack mismatch")
            if os.environ.get("MODAL_PROFILE") != auth["workspace_receipt"]["modal_profile"]:
                raise SweepError("MODAL_PROFILE mismatch")
        except Exception as exc:
            blockers.append(f"authorization_invalid:{type(exc).__name__}")
    return {"version": VERSION, "status": "READY_TO_SPAWN_ONE_DETACHED_CAMPAIGN" if not blockers else "NO_GO", "launch_ready": not blockers, "blockers": blockers, "run_id": auth.get("run_id") if auth else None, "campaign_rounds": campaign_rounds(), "maximum_total_compute_charge_usd": MAXIMUM_TOTAL_COMPUTE_CHARGE_USD, "minimum_outside_credit_reserve_usd": MINIMUM_OUTSIDE_RESERVE_USD, "paid_action_performed": False}


def emit_safe_commands(acceptance_path: Path, output: Path = DEFAULT_COMMANDS) -> dict[str, Any]:
    acceptance = read_json(acceptance_path)
    if not (valid_contract(acceptance, "acceptance_id") and acceptance.get("executor_sha256") == sha_file(Path(__file__)) and acceptance.get("manifest_sha256") == MANIFEST_SHA256 and acceptance.get("spec_sha256") == SPEC_SHA256 and all(acceptance.get("checks", {}).values())):
        raise SweepError("static acceptance is stale or invalid")
    module = "pipeline/modal_train_gemma4_e4b_step50_sweep_campaign_v1.py"
    value = content_contract({"version": VERSION, "status": "SAFE_COMMAND_TEMPLATES_ONLY_NO_AUTHORIZATION_NO_LAUNCH", "acceptance_id": acceptance["acceptance_id"], "mint_template": f"python -m pipeline.modal_train_gemma4_e4b_step50_sweep_campaign_v1 --mint-authorization --acceptance {acceptance_path} --workspace-receipt <fresh.json> --run-id <run_id> --output <authorization.json>", "detached_launch_template": f"$env:MODAL_PROFILE='<bound_profile>'; $env:BICHAR_MODAL_DETACHED='1'; modal run --detach {module}::launch_one --run-id <run_id> --authorization-path <authorization.json> --detached-attestation", "exactly_one_paid_call": True, "maximum_total_compute_charge_usd": MAXIMUM_TOTAL_COMPUTE_CHARGE_USD, "minimum_outside_credit_reserve_usd": MINIMUM_OUTSIDE_RESERVE_USD, "paid_action_performed": False}, "commands_id")
    atomic_json(output, value)
    return value


def _build_image() -> modal.Image:
    bundle = load_mixed_pack(verify=True)
    image = (modal.Image.debian_slim(python_version="3.11")
             .pip_install(*guarded.factorial.FACTORIAL_ENVIRONMENT_PACKAGES)
             .env({"PYTHONPATH": "/root", "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True"})
             .add_local_file(str(Path(__file__)), remote_path="/root/pipeline/modal_train_gemma4_e4b_step50_sweep_campaign_v1.py")
             .add_local_file(str(Path(__file__)), remote_path="/contracts/sweep_executor.py")
             .add_local_file(str(Path(guarded.__file__)), remote_path="/root/pipeline/modal_train_gemma4_e4b_step50_guarded_continuation_v1.py")
             .add_local_file(str(Path(guarded.factorial.__file__)), remote_path="/root/pipeline/modal_train_gemma4_e4b_contextual_repair_factorial_canary_v1.py")
             .add_local_file(str(Path(guarded.proven.__file__)), remote_path="/root/pipeline/modal_train_gemma4_e4b_contextual_heavy14m_v1.py")
             .add_local_file(str(guarded.factorial.MODEL_MANIFEST), remote_path="/contracts/model_manifest.json")
             .add_local_file(str(guarded.factorial.STAGE_RECEIPT), remote_path="/contracts/stage_receipt.json")
             .add_local_file(str(guarded.factorial.TOKENIZER_RECEIPT), remote_path="/contracts/tokenizer_receipt.json")
             .add_local_file(str(MANIFEST), remote_path="/sweep/manifest.json")
             .add_local_file(str(SPEC), remote_path="/sweep/sweep_spec.json"))
    for filename, spec in bundle["manifest"]["artifacts"].items():
        if filename in PACK_FILES.values():
            image = image.add_local_file(str(_artifact_path(spec)), remote_path=f"/sweep/{filename}")
    for role, spec in bundle["manifest"]["panel_bindings"].items():
        if role in PANEL_ROLES:
            image = image.add_local_file(str(_artifact_path(spec)), remote_path=f"/panels/{role}.jsonl")
    return image


def _select_image() -> modal.Image:
    return _build_image() if modal.is_local() else modal.Image.debian_slim(python_version="3.11")


def sweep_causal_loss(logits: Any, labels: Any, spans: Sequence[tuple[int, int]], verdict_mass: float) -> Any:
    import torch
    import torch.nn.functional as F
    shifted_logits, shifted_labels = logits[:, :-1, :], labels[:, 1:]
    valid = shifted_labels.ne(-100)
    parent_losses = []
    for index, span in enumerate(spans):
        row_logits, row_labels = shifted_logits[index, valid[index], :], shifted_labels[index, valid[index]]
        if row_labels.numel() == 0:
            raise SweepError("training parent has no supervised tokens")
        losses = F.cross_entropy(row_logits.float(), row_labels, reduction="none")
        weights = torch.tensor(guarded.factorial.verdict_weight_vector(int(losses.shape[0]), *span, verdict_mass=verdict_mass), dtype=losses.dtype, device=losses.device)
        parent_losses.append((losses * weights).sum())
    return torch.stack(parent_losses).mean()


def _remote_bundle(auth: Mapping[str, Any]) -> dict[str, Any]:
    manifest = read_json(Path("/sweep/manifest.json"), MANIFEST_SHA256)
    spec = read_json(Path("/sweep/sweep_spec.json"), SPEC_SHA256)
    if not (valid_contract(manifest, "manifest_id") and manifest["manifest_id"] == MANIFEST_ID and valid_contract(spec, "spec_id") and spec["spec_id"] == SPEC_ID):
        raise SweepError("remote pack identity drifted")
    rows: dict[str, list[dict[str, Any]]] = {}
    for role, filename in PACK_FILES.items():
        path = Path("/sweep") / filename
        binding = auth["artifact_bindings"][filename]
        if sha_file(path) != binding["sha256"] or binding["sha256"] != manifest["artifacts"][filename]["sha256"]:
            raise SweepError(f"remote artifact drifted: {filename}")
        rows[role] = read_jsonl(path)
        if len(rows[role]) != int(binding["rows"]):
            raise SweepError(f"remote artifact count drifted: {filename}")
    panels = {}
    for role in PANEL_ROLES:
        path = Path("/panels") / f"{role}.jsonl"
        binding = auth["panel_bindings"][role]
        if sha_file(path) != binding["sha256"] or binding["sha256"] != manifest["panel_bindings"][role]["sha256"]:
            raise SweepError(f"remote panel drifted: {role}")
        panels[role] = read_jsonl(path)
    return {"manifest": manifest, "spec": spec, "rows": rows, "panels": panels}


def _atomic_checkpoint(model: Any, optimizer: Any, run_root: Path, arm: str, step: int, ledger: Mapping[str, Any], started: float) -> tuple[Path, dict[str, Any]]:
    import torch
    final = run_root / "arms" / arm / "checkpoints" / f"step-{step:07d}"
    staging = final.parent / f".step-{step:07d}.staging-{uuid.uuid4().hex}"
    staging.mkdir(parents=True, exist_ok=False)
    try:
        model.save_pretrained(staging / "adapter", safe_serialization=True)
        torch.save({"optimizer": optimizer.state_dict(), "python_rng": random.getstate(), "torch_rng": torch.get_rng_state(), "cuda_rng": torch.cuda.get_rng_state_all(), "optimizer_step": step}, staging / "optimizer_rng.pt")
        atomic_json(staging / "ledger.json", dict(ledger))
        tree = guarded.tree_binding(staging, exclude=frozenset({"complete.json"}))
        marker = content_contract({"version": VERSION, "status": "atomic_sweep_arm_checkpoint_complete", "arm": arm, "optimizer_step": step, "manifest_id": MANIFEST_ID, "spec_id": SPEC_ID, "source_adapter_tree_sha256": guarded.SOURCE_ADAPTER_TREE_SHA256, "adapter_tree_sha256": guarded.tree_binding(staging / "adapter")["tree_sha256"], "checkpoint_tree_sha256": tree["tree_sha256"], "elapsed_seconds": time.perf_counter() - started, "promotion_allowed": False}, "marker_id")
        atomic_json(staging / "complete.json", marker)
        final.parent.mkdir(parents=True, exist_ok=True)
        os.replace(staging, final)
        output_volume.commit()
        return final, marker
    except BaseException:
        shutil.rmtree(staging, ignore_errors=True)
        raise


def _atomic_evaluation(model: Any, tokenizer: Any, bundle: Mapping[str, Any], run_root: Path, arm: str, step: int, marker: Mapping[str, Any], deadline: float) -> dict[str, Any]:
    final = run_root / "arms" / arm / "evaluations" / f"step-{step:07d}"
    staging = final.parent / f".step-{step:07d}.staging-{uuid.uuid4().hex}"
    staging.mkdir(parents=True, exist_ok=False)
    try:
        result = guarded._augment_evaluation(guarded.proven._score_milestone(model, tokenizer, bundle["panels"], step, deadline), bundle["panels"])
        summary = content_contract({"version": VERSION, "status": "locked_sweep_arm_evaluation_complete", "arm": arm, "optimizer_step": step, "checkpoint_marker_id": marker["marker_id"], "temperature": result["temperature"], "calibrated_panels": result["calibrated_panels"], "raw_temperature1_panels": result["raw_temperature1_panels"], "runtime_seconds": result["runtime_seconds"], "rank": list(aggregate_rank({"optimizer_step": step, "calibrated_panels": result["calibrated_panels"]})), "private_labels_uploaded": False, "promotion_allowed": False}, "evaluation_id")
        for role in PANEL_ROLES:
            guarded.write_jsonl(staging / f"{role}.jsonl", result["constrained_score_rows"][role])
        atomic_json(staging / "summary.json", summary)
        final.parent.mkdir(parents=True, exist_ok=True)
        os.replace(staging, final)
        output_volume.commit()
        return summary
    except BaseException:
        shutil.rmtree(staging, ignore_errors=True)
        raise


def _load_arm(source_adapter: Path, checkpoint: Path | None, arm: str, tokenizer: Any) -> tuple[Any, Any, list[Any], int, dict[str, Any]]:
    import torch
    from peft import PeftModel, prepare_model_for_kbit_training
    from transformers import AutoModelForMultimodalLM, BitsAndBytesConfig
    # Re-prove the exact immutable offline model tree before every one of the
    # at-most-seven isolated base reloads in this single paid container.
    guarded.proven._remote_model_tree()
    random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
    quantization = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.bfloat16)
    base = AutoModelForMultimodalLM.from_pretrained(guarded.proven.REMOTE_MODEL_DIR, local_files_only=True, trust_remote_code=False, quantization_config=quantization, device_map={"": 0}, dtype=torch.bfloat16, low_cpu_mem_usage=True)
    base.config.use_cache = False
    if hasattr(base.config, "text_config"): base.config.text_config.use_cache = False
    matched = [name for name, _ in base.named_modules() if re.fullmatch(guarded.proven.TARGET_MODULES_REGEX, name)]
    if len(matched) != guarded.proven.EXPECTED_TARGET_MODULES:
        raise SweepError("LoRA topology drifted")
    base = prepare_model_for_kbit_training(base, use_gradient_checkpointing=True)
    model = PeftModel.from_pretrained(base, str((checkpoint / "adapter") if checkpoint else source_adapter), is_trainable=True)
    model.gradient_checkpointing_enable()
    trainable = [parameter for parameter in model.parameters() if parameter.requires_grad]
    if len(trainable) != guarded.proven.EXPECTED_LORA_TENSORS or sum(p.numel() for p in trainable) != guarded.proven.EXPECTED_TRAINABLE_PARAMETERS:
        raise SweepError("trainable adapter topology drifted")
    optimizer = torch.optim.AdamW(trainable, lr=ARMS[arm]["learning_rate"] / WARMUP_END, betas=(0.9, .999), weight_decay=.01)
    ledger: dict[str, Any] = {"loss": [], "learning_rate": [], "gradient": [], "quality": [], "fresh_optimizer_each_arm": True}
    start_step = 0
    if checkpoint:
        state = torch.load(checkpoint / "optimizer_rng.pt", map_location="cpu", weights_only=False)
        optimizer.load_state_dict(state["optimizer"]); start_step = int(state["optimizer_step"])
        random.setstate(state["python_rng"]); torch.set_rng_state(state["torch_rng"]); torch.cuda.set_rng_state_all(state["cuda_rng"])
        ledger = read_json(checkpoint / "ledger.json")
    return model, optimizer, trainable, start_step, ledger


def _run_arm_phase(*, arm: str, target_step: int, bundle: Mapping[str, Any], tokenizer: Any, source_adapter: Path, run_root: Path, started: float, deadline: float, baseline: Mapping[str, Any] | None, best: Mapping[str, Any] | None, checkpoint: Path | None) -> tuple[dict[str, Any], dict[str, Any], Path, bool]:
    import torch
    if time.perf_counter() >= deadline - 150:
        raise SweepError("campaign reserve reached before isolated base reload")
    model, optimizer, trainable, cursor, ledger = _load_arm(source_adapter, checkpoint, arm, tokenizer)
    encoded = {str(row["example_id"]): row for row in bundle["rows"]["train_encoded"]}
    spans = {str(row["example_id"]): (int(row["verdict_value_span_start"]), int(row["verdict_value_span_end"])) for row in bundle["rows"]["loss_weight_spans"]}
    stopped = False
    soft_count = 0
    for prior in reversed(ledger.get("quality", [])):
        if not prior.get("material_soft_regression"):
            break
        soft_count += 1
    try:
        if cursor == 0:
            checkpoint, marker = _atomic_checkpoint(model, optimizer, run_root, arm, 0, ledger, started)
            evaluation = _atomic_evaluation(model, tokenizer, bundle, run_root, arm, 0, marker, deadline)
            baseline = evaluation; best = evaluation
            atomic_json(run_root / "arms" / arm / "best-checkpoint.json", content_contract({"version": VERSION, "arm": arm, "optimizer_step": 0, "evaluation_id": evaluation["evaluation_id"], "checkpoint_marker_id": marker["marker_id"], "rank": list(aggregate_rank(evaluation)), "promotion_allowed": False}, "best_id")); output_volume.commit()
        for position in range(cursor, target_step):
            if time.perf_counter() >= deadline - 90:
                raise SweepError("campaign cost/time reserve reached")
            step_number = position + 1
            row_ids = [str(value) for value in bundle["rows"]["optimizer_steps"][position]["example_ids"]]
            rows = [encoded[value] for value in row_ids]
            for group in optimizer.param_groups: group["lr"] = wsd_learning_rate(step_number, ARMS[arm]["learning_rate"])
            optimizer.zero_grad(set_to_none=True); losses = []
            for offset in (0, 4):
                batch = guarded.proven._collate(rows[offset:offset + 4], tokenizer.pad_token_id); labels = batch.pop("labels")
                with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
                    loss = sweep_causal_loss(model(**batch).logits, labels, [spans[value] for value in row_ids[offset:offset + 4]], ARMS[arm]["verdict_loss_mass"])
                if not torch.isfinite(loss): raise SweepError("nonfinite loss")
                (loss / 2).backward(); losses.append(float(loss.detach().cpu()))
            gradient = float(torch.nn.utils.clip_grad_norm_(trainable, 1.0).detach().cpu())
            if not math.isfinite(gradient): raise SweepError("nonfinite gradient")
            optimizer.step(); cursor = step_number
            ledger["loss"].append({"optimizer_step": cursor, "mean": statistics.fmean(losses), "verdict_mass": ARMS[arm]["verdict_loss_mass"]})
            ledger["learning_rate"].append({"optimizer_step": cursor, "learning_rate": wsd_learning_rate(cursor, ARMS[arm]["learning_rate"])})
            ledger["gradient"].append({"optimizer_step": cursor, "preclip_l2_norm": gradient})
            if cursor % 10 == 0:
                checkpoint, marker = _atomic_checkpoint(model, optimizer, run_root, arm, cursor, ledger, started)
                evaluation = _atomic_evaluation(model, tokenizer, bundle, run_root, arm, cursor, marker, deadline)
                assert baseline is not None and best is not None
                catastrophic = guarded.catastrophic_regression_gate(baseline, evaluation)
                if not catastrophic["all_gates_passed"]:
                    stopped = True
                elif cursor > 20 and material_aggregate_regression(best, evaluation):
                    soft_count += 1; stopped = soft_count >= SOFT_PATIENCE
                else:
                    soft_count = 0
                ledger["quality"].append({"optimizer_step": cursor, "catastrophic": catastrophic, "material_soft_regression": material_aggregate_regression(best, evaluation), "consecutive": soft_count, "stopped": stopped})
                if eligible_best_update(best, evaluation, catastrophic, stopped):
                    best = evaluation
                    atomic_json(run_root / "arms" / arm / "best-checkpoint.json", content_contract({"version": VERSION, "arm": arm, "optimizer_step": cursor, "evaluation_id": evaluation["evaluation_id"], "checkpoint_marker_id": marker["marker_id"], "rank": list(aggregate_rank(evaluation)), "promotion_allowed": False}, "best_id")); output_volume.commit()
                if stopped: break
                model.train()
        if checkpoint is None:
            raise SweepError("arm produced no atomic checkpoint")
        atomic_json(run_root / "arms" / arm / "restart-ledger.json", content_contract({"version": VERSION, "arm": arm, "last_atomic_step": cursor, "checkpoint": str(checkpoint.relative_to(run_root)), "stopped": stopped, "restart_automatic": False}, "restart_id")); output_volume.commit()
        return dict(baseline), dict(best), checkpoint, stopped
    finally:
        del optimizer, trainable, model
        gc.collect()
        torch.cuda.empty_cache()


def _remote_campaign(run_id: str, authorization: Mapping[str, Any]) -> dict[str, Any]:
    """Execute the sealed campaign; expensive work is intentionally isolated here.

    Base weights are reloaded from the same hash-bound offline snapshot for each
    arm/phase.  This is the sealed fail-closed alternative to relying on a weak
    in-memory base-invariance checksum and prevents cross-arm contamination.
    """
    executor_sha = sha_file(Path("/contracts/sweep_executor.py"))
    auth = _validate_authorization(authorization, run_id, time.time(), executor_sha)
    bundle = _remote_bundle(auth)
    import torch
    from transformers import AutoProcessor
    if not (torch.cuda.is_available() and torch.cuda.device_count() == 1 and "H200" in torch.cuda.get_device_name(0).upper() and torch.cuda.is_bf16_supported()):
        raise SweepError("exactly one bf16 H200 required")
    started = time.perf_counter()
    # Conservative aggregate cap; includes a persistence tail.
    usable = min(HARD_WALL_SECONDS, (MAXIMUM_TOTAL_COMPUTE_CHARGE_USD - .15) / guarded.TOTAL_RATE_USD_PER_SECOND)
    deadline = started + usable
    consumption = Path("/sweep_output/authorizations") / auth["authorization_id"]
    consumption.mkdir(parents=True, exist_ok=False)
    atomic_json(consumption / "consumed.json", content_contract({"version": VERSION, "run_id": run_id, "consumed_before_model_load": True}, "consumption_id")); output_volume.commit()
    guarded.proven._remote_model_tree()
    source_adapter = guarded._verify_source_step50()
    processor = AutoProcessor.from_pretrained(guarded.proven.REMOTE_MODEL_DIR, local_files_only=True, trust_remote_code=False)
    tokenizer = processor.tokenizer
    if tokenizer.convert_tokens_to_ids(guarded.TARGET_TERMINATOR_TOKEN) != guarded.TARGET_TERMINATOR_ID or tokenizer.pad_token_id != 0:
        raise SweepError("offline tokenizer identity drifted")
    for raw, encoded in zip(bundle["rows"]["train_examples"], bundle["rows"]["train_encoded"], strict=True):
        if any(encoded.get(field) != value for field, value in guarded.proven._reencode(tokenizer, raw).items()):
            raise SweepError(f"chat replay drifted: {raw['example_id']}")
    run_root = Path("/sweep_output/runs") / run_id
    run_root.mkdir(parents=True, exist_ok=False)
    states: dict[str, dict[str, Any]] = {}
    for arm in sorted(ARMS):
        baseline, best, checkpoint, stopped = _run_arm_phase(arm=arm, target_step=20, bundle=bundle, tokenizer=tokenizer, source_adapter=source_adapter, run_root=run_root, started=started, deadline=deadline, baseline=None, best=None, checkpoint=None)
        states[arm] = {"baseline": baseline, "best": best, "checkpoint": checkpoint, "stopped": stopped}
    live = [arm for arm in sorted(ARMS) if not states[arm]["stopped"]]
    if len(live) < 2:
        raise SweepError("fewer than two arms survived the exact step20 screen")
    top2 = sorted(live, key=lambda arm: aggregate_rank(states[arm]["best"]), reverse=True)[:2]
    atomic_json(run_root / "selection-step20.json", content_contract({"version": VERSION, "selected_top2": top2, "ranks": {arm: list(aggregate_rank(states[arm]["best"])) for arm in states}, "best_not_last": True}, "selection_id")); output_volume.commit()
    for arm in top2:
        state = states[arm]
        baseline, best, checkpoint, stopped = _run_arm_phase(arm=arm, target_step=60, bundle=bundle, tokenizer=tokenizer, source_adapter=source_adapter, run_root=run_root, started=started, deadline=deadline, baseline=state["baseline"], best=state["best"], checkpoint=state["checkpoint"])
        states[arm] = {"baseline": baseline, "best": best, "checkpoint": checkpoint, "stopped": stopped}
    live2 = [arm for arm in top2 if not states[arm]["stopped"]]
    if not live2: raise SweepError("all top-two arms stopped before winner selection")
    winner = max(live2, key=lambda arm: aggregate_rank(states[arm]["best"]))
    atomic_json(run_root / "selection-step60.json", content_contract({"version": VERSION, "winner": winner, "top2": top2, "ranks": {arm: list(aggregate_rank(states[arm]["best"])) for arm in top2}, "best_not_last": True}, "selection_id")); output_volume.commit()
    state = states[winner]
    baseline, best, checkpoint, stopped = _run_arm_phase(arm=winner, target_step=100, bundle=bundle, tokenizer=tokenizer, source_adapter=source_adapter, run_root=run_root, started=started, deadline=deadline, baseline=state["baseline"], best=state["best"], checkpoint=state["checkpoint"])
    elapsed = time.perf_counter() - started
    bounded_cost = (elapsed + 30.0) * guarded.TOTAL_RATE_USD_PER_SECOND
    if bounded_cost > MAXIMUM_TOTAL_COMPUTE_CHARGE_USD:
        raise SweepError("aggregate campaign cost cap exceeded")
    terminal = content_contract({"version": VERSION, "status": "sweep_campaign_complete" if not stopped else "winner_stopped_by_guard", "run_id": run_id, "top2": top2, "winner": winner, "best_optimizer_step": best["optimizer_step"], "best_evaluation_id": best["evaluation_id"], "best_rank": list(aggregate_rank(best)), "elapsed_seconds": elapsed, "bounded_total_compute_charge_usd": bounded_cost, "maximum_total_compute_charge_usd": MAXIMUM_TOTAL_COMPUTE_CHARGE_USD, "private_labels_used": False, "promotion_allowed": False}, "terminal_id")
    atomic_json(run_root / "terminal.json", terminal); output_volume.commit()
    return terminal


app = modal.App("bichar-gemma4-e4b-step50-sweep-campaign-v1")
model_volume = modal.Volume.from_name(guarded.MODEL_VOLUME, create_if_missing=False)
source_volume = modal.Volume.from_name(guarded.SOURCE_CHECKPOINT_VOLUME, create_if_missing=False)
output_volume = modal.Volume.from_name(OUTPUT_VOLUME, create_if_missing=True)


@app.function(image=_select_image(), gpu="H200", cpu=8, memory=65_536,
              timeout=HARD_WALL_SECONDS, retries=0, block_network=True,
              restrict_modal_access=True, single_use_containers=True,
              max_containers=1,
              volumes={"/models": model_volume.read_only(), "/step50_source": source_volume.read_only(), "/sweep_output": output_volume})
def train_step50_sweep_campaign_h200(run_id: str, authorization: dict[str, Any]) -> dict[str, Any]:
    try:
        return _remote_campaign(run_id, authorization)
    except BaseException as exc:
        failure = content_contract({"version": VERSION, "status": "fatal_sweep_campaign_failure", "run_id": run_id, "exception_type": type(exc).__name__, "exception_message": str(exc), "traceback": traceback.format_exc(), "promotion_allowed": False}, "failure_id")
        atomic_json(Path("/sweep_output/failures") / f"{failure['failure_id']}.json", failure)
        output_volume.commit()
        raise


@app.local_entrypoint()
def launch_one(run_id: str, authorization_path: str, detached_attestation: bool = False) -> None:
    if not detached_attestation or os.environ.get("BICHAR_MODAL_DETACHED") != "1":
        raise SweepError("detached launch attestation missing")
    plan = build_local_plan(DEFAULT_ACCEPTANCE, Path(authorization_path))
    if not plan["launch_ready"] or plan["run_id"] != run_id:
        raise SweepError(f"campaign plan is NO-GO: {plan['blockers']}")
    authorization = read_json(Path(authorization_path))
    call = train_step50_sweep_campaign_h200.spawn(run_id=run_id, authorization=authorization)
    print(canonical({"status": "exactly_one_detached_h200_sweep_campaign_spawned", "run_id": run_id, "call_id": call.object_id, "second_call_spawned": False, "local_tether_waited": False}))


def main(argv: Sequence[str] | None = None) -> int:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--plan", action="store_true")
    parser.add_argument("--write-static-acceptance", action="store_true")
    parser.add_argument("--dry-remote-simulation", action="store_true")
    parser.add_argument("--emit-safe-commands", action="store_true")
    parser.add_argument("--acceptance", type=Path)
    parser.add_argument("--authorization", type=Path)
    parser.add_argument("--workspace-receipt", type=Path)
    parser.add_argument("--run-id")
    parser.add_argument("--mint-authorization", action="store_true")
    parser.add_argument("--output", type=Path)
    args = parser.parse_args(argv)
    if args.dry_remote_simulation:
        value = dry_remote_simulation()
    elif args.emit_safe_commands:
        if args.acceptance is None:
            raise SweepError("--acceptance is required")
        value = emit_safe_commands(args.acceptance, args.output or DEFAULT_COMMANDS)
    elif args.write_static_acceptance:
        value = static_acceptance(); atomic_json(args.output or DEFAULT_ACCEPTANCE, value)
    elif args.mint_authorization:
        if not all((args.acceptance, args.workspace_receipt, args.run_id, args.output)):
            raise SweepError("minting requires acceptance, workspace receipt, run ID, output")
        value = mint_authorization(args.acceptance, args.workspace_receipt, args.run_id); atomic_json(args.output, value)
    else:
        value = build_local_plan(args.acceptance, args.authorization)
    print(json.dumps(value, ensure_ascii=False, sort_keys=True, indent=2))
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


## Measured learning curve

The values below are copied from immutable evaluation receipts. Different
evaluation panels are not pooled with Challenge350 or the public leaderboard.

In [ ]:
records = [
    {"step": 0,  "group_source_mean": 0.857434, "long_f1": 0.324444},
    {"step": 10, "group_source_mean": 0.923684, "long_f1": 0.570370},
    {"step": 20, "group_source_mean": 0.907835, "long_f1": 0.444444},
    {"step": 30, "group_source_mean": 0.907835, "long_f1": 0.389404},
    {"step": 40, "group_source_mean": 0.874395, "long_f1": 0.324444},
]

try:
    import pandas as pd
    import matplotlib.pyplot as plt
    from IPython.display import display

    frame = pd.DataFrame(records)
    display(frame)
    axis = frame.plot(
        x="step",
        y=["group_source_mean", "long_f1"],
        marker="o",
        ylim=(0.25, 1.0),
        title="Gemma 4 E4B guarded continuation: best checkpoint is not last",
    )
    axis.axvline(10, color="#D99C18", linestyle="--", label="retained step")
    axis.set_ylabel("macro-F1")
    axis.grid(alpha=0.25)
    plt.show()
except ImportError:
    print(json.dumps(records, indent=2))

## Interpretation

The adapter learned useful evidence-verdict behavior quickly, especially on
the long-context diagnostic, but continued optimization erased that gain. An
earlier high-learning-rate run also collapsed after roughly two million tokens.
Therefore the production pipeline treats the model as a selective residual
reasoner rather than the whole system.

The bundled GitHub repository includes the guarded continuation executor,
quality-pilot executor, environment pins, model specification, curriculum
builder, and measured score ledger. Private rows, private labels, weights, API
tokens, and paid-run authorizations are excluded.